# Train only the Rater — EA-style gate + Waterbirds-ERM embeddings

This notebook trains **only the Rater**. The final classifier is intentionally postponed to a later notebook.

Representation:
- frozen **Waterbirds-ERM ResNet-50**;
- the already-saved 2048-D ERM embeddings on Google Drive are reused;
- the ERM classifier head is **not** used to initialize the inner models.

Inner models:
- two independent random-from-scratch `Linear(2048,2)` classifiers;
- persistent for all 500 meta steps;
- no refreshing/reinitialization.

The weighting/loss rule matches the supplied Rater implementation:

```text
correct prediction       -> weight = 1
misclassified prediction -> weight = RaterRating(z)
```

and

```text
L_inner = SUM_i weight_i * CE_i
```

with **no division by `sum(weights)`**.

Before **every** Rater update, both inner classifiers each see the complete `train_no_aug` dataset exactly once. With 4,795 examples and batch size 128, that is 38 differentiable inner steps per model before one outer update.

The Rater meta-loss is classification cross-entropy only. No gradient auxiliary loss is used.

Saved to Drive:

```text
plots/
log.txt
summary.txt
final_rater.pt
```

In [1]:
# 1. Paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime
import os, sys, subprocess, shutil, re, torch

DRIVE_ROOT = Path("/content/drive/MyDrive/evidential_alignment_colab")
REPO_DIR = Path("/content/evidential_alignment")
DATA_ROOT = Path("/content/datasets")
WB_DIR = DATA_ROOT / "waterbird_complete95_forest2water2"
WB_ARCHIVE = DRIVE_ROOT / "archives" / "waterbird_complete95_forest2water2.tar.gz"

ERM_MODEL = DRIVE_ROOT / "erm_base_models" / "waterbirds_erm_resnet50_imagenet_100ep_best_val_acc.pt"
if not ERM_MODEL.exists():
    ERM_MODEL = DRIVE_ROOT / "erm_base_models" / "erm_waterbirds_resnet50_train_128B_100E_seed0_pretrained" / "best_val_acc_model.pt"
assert ERM_MODEL.exists(), ERM_MODEL

EMBED_ROOT = DRIVE_ROOT / "embeddings_waterbirds_erm"
OUTPUT_ROOT = DRIVE_ROOT / "rater_meta_only_runs"
LOCAL_PRE = Path("/content/rater_precompute_local")
LOCAL_RUN = Path("/content/rater_meta_only_local")

for p in [WB_ARCHIVE.parent, EMBED_ROOT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("ERM:", ERM_MODEL)
print("Embedding cache:", EMBED_ROOT)
print("Rater outputs:", OUTPUT_ROOT)

Mounted at /content/drive
ERM: /content/drive/MyDrive/evidential_alignment_colab/erm_base_models/waterbirds_erm_resnet50_imagenet_100ep_best_val_acc.pt
Embedding cache: /content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm
Rater outputs: /content/drive/MyDrive/evidential_alignment_colab/rater_meta_only_runs


In [2]:
# 2. Fresh repo + dependencies
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone",
    "https://github.com/TheRealMmd/evidential_alignment.git",
    str(REPO_DIR),
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "timm", "transformers", "wilds", "wget", "matplotlib", "scipy",
], check=True)

print("✓ repo ready")

✓ repo ready


In [3]:
# 3. Waterbirds
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not (WB_ARCHIVE.exists() and WB_ARCHIVE.stat().st_size > 100_000_000):
    subprocess.run([
        "wget", "-c",
        "https://downloads.cs.stanford.edu/nlp/data/dro/waterbird_complete95_forest2water2.tar.gz",
        "-O", str(WB_ARCHIVE),
    ], check=True)

if not (WB_DIR / "metadata.csv").exists():
    subprocess.run([
        "tar", "-xzf", str(WB_ARCHIVE), "-C", str(DATA_ROOT)
    ], check=True)

assert (WB_DIR / "metadata.csv").exists()
print("✓ Waterbirds ready")

✓ Waterbirds ready


In [4]:
# 4. Patch data path + PyTorch 2.6 loading
config_path = REPO_DIR / "config.py"
text = config_path.read_text(encoding="utf-8")
placeholder = '"waterbirds": "/path/to/datasets/waterbirds"'
if placeholder in text:
    text = text.replace(placeholder, f'"waterbirds": "{WB_DIR}"')
else:
    text, n = re.subn(
        r'("waterbirds"\s*:\s*)["\'][^"\']*["\']',
        lambda m: m.group(1) + '"' + str(WB_DIR) + '"',
        text,
        count=1,
    )
    assert n == 1
config_path.write_text(text, encoding="utf-8")

algorithm_path = REPO_DIR / "algorithms" / "algorithm.py"
text = algorithm_path.read_text(encoding="utf-8")
text = text.replace(
    'torch.load(check_point, map_location="cpu")',
    'torch.load(check_point, map_location="cpu", weights_only=False)',
)
text = text.replace(
    "torch.load(check_point, map_location='cpu')",
    'torch.load(check_point, map_location="cpu", weights_only=False)',
)
algorithm_path.write_text(text, encoding="utf-8")
print("✓ repo patched")

✓ repo patched


In [5]:
# 5. Install updated rater.py
import ast
rater_path = REPO_DIR / "algorithms" / "rater.py"
RATER_SOURCE = 'import os\nimport csv\nimport hashlib\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom scipy import stats\nfrom torch.utils.data import Dataset, DataLoader\nfrom tqdm import tqdm\n\nfrom .algorithm import Algorithm\nfrom .register import register_algorithm\n\nfrom models.classifier import Classifier\nfrom utils import log\n\n\n# ============================================================\n# Embedding dataset\n# ============================================================\n\nclass EmbeddingTensorDataset(Dataset):\n    """Dataset of frozen backbone embeddings and Waterbirds metadata."""\n\n    def __init__(self, embeddings, labels, groups, attrs):\n        self.embeddings = embeddings.float()\n        self.y_array = labels.long()\n        self.group_array = groups.long()\n        self.confounder_array = attrs.long()\n        self.n_classes = int(torch.unique(self.y_array).numel())\n\n    def __len__(self):\n        return len(self.y_array)\n\n    def __getitem__(self, idx):\n        return (\n            self.embeddings[idx],\n            self.y_array[idx],\n            self.group_array[idx],\n            self.confounder_array[idx],\n        )\n\n\n# ============================================================\n# Rater architectures\n# ============================================================\n\nclass EmbeddingRaterSmall(nn.Module):\n    """Small MLP rater for a ResNet-50 pooled embedding."""\n\n    def __init__(self, input_dim=2048):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.LayerNorm(input_dim),\n            nn.Linear(input_dim, 256),\n            nn.ReLU(inplace=True),\n            nn.Linear(256, 64),\n            nn.ReLU(inplace=True),\n            nn.Linear(64, 1),\n        )\n\n    def forward(self, z):\n        return self.net(z).squeeze(-1)\n\n\nclass EmbeddingRaterMedium(nn.Module):\n    """Medium-capacity MLP rater for a ResNet-50 pooled embedding."""\n\n    def __init__(self, input_dim=2048):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.LayerNorm(input_dim),\n            nn.Linear(input_dim, 512),\n            nn.GELU(),\n            nn.Dropout(0.1),\n            nn.Linear(512, 256),\n            nn.GELU(),\n            nn.Dropout(0.1),\n            nn.Linear(256, 64),\n            nn.GELU(),\n            nn.Linear(64, 1),\n        )\n\n    def forward(self, z):\n        return self.net(z).squeeze(-1)\n\n\n# ============================================================\n# Inner / downstream classifier\n# ============================================================\n\nclass InnerLinearClassifier(nn.Module):\n    """Linear classifier operating on frozen embeddings."""\n\n    def __init__(self, input_dim, num_classes):\n        super().__init__()\n        self.linear = nn.Linear(input_dim, num_classes)\n\n    def forward(self, z):\n        return self.linear(z)\n\n\n# ============================================================\n# Rater algorithm\n# ============================================================\n\n@register_algorithm("rater")\nclass Rater(Algorithm):\n    """\n    Frozen trained feature-Rater used only for final-classifier training.\n\n    Image x -> frozen Waterbirds-ERM ResNet-50 -> z -> r_eta(z) -> scalar score.\n\n    Inner update:\n\n        First, the CURRENT inner classifier predicts each sample.\n\n        If the prediction is CORRECT:\n            w_i = 1\n\n        If the prediction is WRONG:\n            the Rater is evaluated ONLY for that misclassified sample,\n            and its raw score is converted to a rating using the selected\n            Rater transform:\n\n            SOFTMAX:\n                r_i = softmax((score_i - mean_misclassified) / tau)\n\n            SIGMOID:\n                standardized_i =\n                    (score_i - mean_misclassified)\n                    / (std_misclassified + eps)\n                r_i = sigmoid(standardized_i / tau)\n\n            Then:\n                w_i = r_i\n\n        Thus the effective weight follows the Evidential-Alignment-style gate:\n\n            w_i = 1                         if prediction is correct\n            w_i = RaterRating(z_i)          if prediction is wrong\n\n        The Rater is NOT used to determine the training weight of correctly\n        classified samples.\n\n        The requested weighted objective is retained:\n\n            L_inner = sum_i w_i CE(h_theta(z_i), y_i)\n\n        There is intentionally NO division by sum_i w_i.\n\n        theta\' = theta - alpha * grad_theta L_inner\n\n    Outer update:\n        L_outer = CE(h_theta\'(z_val), y_val)\n        eta <- eta - beta * grad_eta L_outer\n\n    In addition to learning the rater, this implementation:\n      * evaluates the persistent inner-model population during meta-training;\n      * logs average accuracy and worst-group accuracy (WGA);\n      * measures correlation between raw/final Rater training weights and loss;\n      * saves per-group histograms for raw Rater scores and final training weights;\n      * saves raw-score-vs-loss and final-weight-vs-loss scatter plots;\n      * optionally saves intermediate rater/inner-model checkpoints;\n      * after meta-training, restores and saves the BEST trained Rater;\n      * this meta-only version does NOT train the final classifier:\n            val_subset1 -> calibration/training,\n            val_subset2 -> model selection,\n            test        -> diagnostic evaluation after every final epoch;\n      * test metrics are NEVER used for checkpoint selection;\n      * trains a fresh final classifier with EA-style dynamic weighting:\n            correct -> 1,\n            misclassified -> trained-Rater rating;\n      * saves the ACTUAL final-classifier weight histograms every epoch;\n      * selects the downstream classifier by validation WGA by default;\n      * evaluates the final classifier on validation/test splits;\n      * uses a persistent shared embedding cache so the frozen ERM ResNet-50\n        embeddings are extracted once and reused across timestamped runs.\n    """\n\n    def __init__(self, config):\n        # The method works on frozen backbone features / last-layer models.\n        config.last_layer = True\n        super().__init__(config)\n\n        if torch.cuda.is_available():\n            self.device = f"cuda:{self.config.gpu}"\n        else:\n            self.device = "cpu"\n\n        self.n_classes = self.datasets["train"].n_classes\n\n        if self.config.backbone != "resnet50":\n            log(\n                f"[Rater] Warning: requested backbone is "\n                f"{self.config.backbone}, not resnet50."\n            )\n\n        if not self.config.pretrained:\n            raise ValueError(\n                "This Rater experiment expects a ResNet-50 that was "\n                "initialized from ImageNet and then ERM-finetuned on "\n                "Waterbirds. Run with --pretrained True."\n            )\n\n        # ----------------------------------------------------\n        # Frozen Waterbirds-ERM feature extractor\n        # ----------------------------------------------------\n        #\n        # IMPORTANT:\n        # We do NOT use a fresh ImageNet-only ResNet-50 here.\n        #\n        # We reconstruct the same Classifier architecture used by the\n        # repository, then load the saved Waterbirds ERM checkpoint:\n        #\n        #   image\n        #       -> ERM-finetuned ResNet-50 backbone\n        #       -> 2048-D embedding z\n        #       -> saved ERM linear classifier\n        #\n        # The backbone is frozen after loading. The saved ERM classifier\n        # parameters are also copied and used to initialize the persistent\n        # inner classifiers for Rater meta-learning.\n        # ----------------------------------------------------\n        self.erm_model_path = str(\n            getattr(\n                self.config,\n                "erm_model",\n                "",\n            )\n            or os.environ.get(\n                "RATER_ERM_MODEL",\n                "",\n            )\n        ).strip()\n\n        if not self.erm_model_path:\n            raise ValueError(\n                "This Rater experiment requires the saved Waterbirds ERM "\n                "checkpoint. Pass it with --erm_model PATH or set "\n                "RATER_ERM_MODEL."\n            )\n\n        if not os.path.exists(self.erm_model_path):\n            raise FileNotFoundError(\n                f"ERM checkpoint does not exist: {self.erm_model_path}"\n            )\n\n        self.feature_model = Classifier(\n            backbone=self.config.backbone,\n            num_classes=self.n_classes,\n            pretrained=True,\n        ).to(self.device)\n\n        erm_checkpoint = torch.load(\n            self.erm_model_path,\n            map_location="cpu",\n            weights_only=False,\n        )\n\n        if (\n            isinstance(erm_checkpoint, dict)\n            and "model_sd" in erm_checkpoint\n        ):\n            erm_state_dict = erm_checkpoint["model_sd"]\n        else:\n            erm_state_dict = erm_checkpoint\n\n        filtered_erm_state_dict = {\n            k: v\n            for k, v in erm_state_dict.items()\n            if not k.endswith("position_ids")\n        }\n\n        incompatible = self.feature_model.load_state_dict(\n            filtered_erm_state_dict,\n            strict=False,\n        )\n\n        self.erm_checkpoint_epoch = (\n            erm_checkpoint.get("epoch", None)\n            if isinstance(erm_checkpoint, dict)\n            else None\n        )\n\n        self.erm_checkpoint_sel_metric = (\n            erm_checkpoint.get("sel_metric", None)\n            if isinstance(erm_checkpoint, dict)\n            else None\n        )\n\n        if not isinstance(\n            self.feature_model.fc,\n            nn.Linear,\n        ):\n            raise TypeError(\n                "Expected the saved ERM Classifier to have a linear .fc "\n                "head, but got "\n                f"{type(self.feature_model.fc).__name__}."\n            )\n\n        self.feature_dim = (\n            self.feature_model.backbone.num_features\n        )\n\n        if (\n            self.feature_model.fc.weight.shape[1]\n            != self.feature_dim\n        ):\n            raise ValueError(\n                "ERM classifier input dimension does not match the "\n                "backbone embedding dimension."\n            )\n\n        # Exact ERM classifier parameters used as the base initialization\n        # for the persistent inner classifiers.\n        self.erm_head_weight = (\n            self.feature_model.fc.weight\n            .detach()\n            .clone()\n        )\n\n        self.erm_head_bias = (\n            self.feature_model.fc.bias\n            .detach()\n            .clone()\n        )\n\n        # Fingerprint the exact checkpoint file so this cache can never be\n        # confused with the old ImageNet-only embedding cache.\n        erm_stat = os.stat(\n            self.erm_model_path\n        )\n\n        fingerprint_text = (\n            f"{os.path.abspath(self.erm_model_path)}|"\n            f"{erm_stat.st_size}|"\n            f"{erm_stat.st_mtime_ns}"\n        )\n\n        self.erm_checkpoint_fingerprint = (\n            hashlib.sha1(\n                fingerprint_text.encode("utf-8")\n            ).hexdigest()[:12]\n        )\n\n        self.feature_model.eval()\n\n        for p in self.feature_model.parameters():\n            p.requires_grad_(False)\n\n        log(\n            f"[Rater] Loaded frozen Waterbirds ERM model from: "\n            f"{self.erm_model_path}"\n        )\n\n        log(\n            f"[Rater] ERM checkpoint epoch="\n            f"{self.erm_checkpoint_epoch}, "\n            f"sel_metric={self.erm_checkpoint_sel_metric}, "\n            f"fingerprint={self.erm_checkpoint_fingerprint}"\n        )\n\n        if incompatible.missing_keys:\n            log(\n                f"[Rater] ERM load missing keys: "\n                f"{incompatible.missing_keys}"\n            )\n\n        if incompatible.unexpected_keys:\n            log(\n                f"[Rater] ERM load unexpected keys: "\n                f"{incompatible.unexpected_keys}"\n            )\n\n        log(\n            f"[Rater] Frozen ERM-finetuned "\n            f"{self.config.backbone} backbone. "\n            f"Embedding dimension = {self.feature_dim}"\n        )\n\n        # ----------------------------------------------------\n        # Persistent SHARED embedding cache\n        # ----------------------------------------------------\n        #\n        # Preferred Colab usage:\n        #\n        #   RATER_EMBEDDING_CACHE_ROOT=\n        #       /content/drive/MyDrive/evidential_alignment_colab/embeddings\n        #\n        # The Rater automatically creates a representation-specific\n        # namespace under that root. The cache is independent of the\n        # timestamped experiment directory, so future runs reuse the same\n        # train/val/test embeddings and do not run ResNet-50 again.\n        # ----------------------------------------------------\n        self.embedding_cache_root = str(\n            getattr(\n                self.config,\n                "rater_embedding_cache_root",\n                "",\n            )\n            or os.environ.get(\n                "RATER_EMBEDDING_CACHE_ROOT",\n                "",\n            )\n        ).strip()\n\n        # One-time notebook preparation mode. When enabled, train() only\n        # builds/validates the persistent embeddings and then returns.\n        self.precompute_embeddings_only = (\n            os.environ.get(\n                "RATER_PRECOMPUTE_EMBEDDINGS_ONLY",\n                "0",\n            ).strip().lower()\n            in {"1", "true", "yes", "y"}\n        )\n\n        # ----------------------------------------------------\n        # Bilevel hyperparameters\n        # ----------------------------------------------------\n        self.meta_steps = int(\n            os.environ.get(\n                "RATER_META_STEPS",\n                str(\n                    getattr(\n                        self.config,\n                        "rater_meta_steps",\n                        self.config.epoch,\n                    )\n                ),\n            )\n        )\n        self.inner_steps = int(\n            getattr(self.config, "rater_inner_steps", 2)\n        )\n        self.num_inner_models = int(\n            os.environ.get(\n                "RATER_NUM_INNER_MODELS",\n                str(\n                    getattr(\n                        self.config,\n                        "rater_num_inner_models",\n                        2,\n                    )\n                ),\n            )\n        )\n        self.inner_lr = float(\n            os.environ.get(\n                "RATER_INNER_LR",\n                str(\n                    getattr(\n                        self.config,\n                        "rater_inner_lr",\n                        1e-2,\n                    )\n                ),\n            )\n        )\n        self.outer_lr = float(\n            os.environ.get(\n                "RATER_OUTER_LR",\n                str(\n                    getattr(\n                        self.config,\n                        "rater_outer_lr",\n                        3e-4,\n                    )\n                ),\n            )\n        )\n        self.temperature = float(\n            os.environ.get(\n                "RATER_TEMPERATURE",\n                str(\n                    getattr(\n                        self.config,\n                        "rater_temperature",\n                        2.0,\n                    )\n                ),\n            )\n        )\n\n        # ----------------------------------------------------\n        # Rater-rating transform for MISCLASSIFIED samples only.\n        #\n        # Correctly classified samples always receive weight = 1.\n        # Misclassified samples receive a Rater-derived rating using the\n        # selected softmax/sigmoid transform.\n        #\n        # Effective training rule:\n        #       weight_i = 1                  if correct\n        #       weight_i = rater_rating_i     if misclassified\n        #\n        # Inner loss remains:\n        #       sum_i weight_i * CE_i\n        #\n        # There is NO division by sum(weights).\n        # ----------------------------------------------------\n        self.weighting = str(\n            os.environ.get(\n                "RATER_WEIGHTING",\n                str(\n                    getattr(\n                        self.config,\n                        "rater_weighting",\n                        "softmax",\n                    )\n                ),\n            )\n        ).lower()\n\n        self.refresh_steps = int(\n            getattr(self.config, "rater_refresh_steps", 100)\n        )\n        self.grad_clip = float(\n            os.environ.get(\n                "RATER_GRAD_CLIP",\n                str(\n                    getattr(\n                        self.config,\n                        "rater_grad_clip",\n                        5.0,\n                    )\n                ),\n            )\n        )\n        self.inner_reg_weight = float(\n            getattr(self.config, "rater_inner_reg_weight", 0.0)\n        )\n        self.outer_reg_weight = 0.0\n        self.score_reg_weight = 0.0\n\n        # ----------------------------------------------------\n        # First ERM-initialized Rater experiment\n        # ----------------------------------------------------\n        #\n        # Rater meta-objective:\n        #\n        #       L_meta = CE(\n        #           h_theta\'(z_outer),\n        #           y_outer\n        #       )\n        #\n        # There is NO gradient-magnitude auxiliary loss and no additional\n        # Rater regularizer in the meta objective for this experiment.\n        #\n        # The inner classifiers start from the SAVED ERM classifier:\n        #\n        #   inner model 0:\n        #       exact ERM classifier weights\n        #\n        #   inner models 1,2,...:\n        #       ERM classifier + very small Gaussian perturbation\n        #\n        # This keeps the two-model population near the same useful ERM\n        # solution without making their meta-trajectories exactly identical.\n        # With RATER_NUM_INNER_MODELS=1, the sole inner model is the exact\n        # saved ERM classifier.\n        # ----------------------------------------------------\n        self.grad_loss_weight = 0.0\n\n        self.inner_init_noise_std = float(\n            os.environ.get(\n                "RATER_INNER_INIT_NOISE_STD",\n                str(\n                    getattr(\n                        self.config,\n                        "rater_inner_init_noise_std",\n                        1e-3,\n                    )\n                ),\n            )\n        )\n\n        if self.inner_init_noise_std < 0:\n            raise ValueError(\n                "RATER_INNER_INIT_NOISE_STD must be >= 0."\n            )\n\n        # This experiment uses the EA-style gate from the supplied Rater:\n        #   correct prediction -> weight 1\n        #   wrong prediction   -> Rater-derived rating\n        self.rate_all_samples = False\n\n        self.rater_capacity = getattr(\n            self.config, "rater_capacity", "medium"\n        )\n\n        # ----------------------------------------------------\n        # Diagnostic plotting. By default, plots are saved at\n        # the same frequency as --eval_freq. If config.py later\n        # defines --rater_plot_freq, it will override eval_freq.\n        # ----------------------------------------------------\n        self.eval_freq = int(\n            os.environ.get(\n                "RATER_EVAL_FREQ",\n                str(\n                    max(\n                        1,\n                        int(\n                            getattr(\n                                self.config,\n                                "eval_freq",\n                                10,\n                            )\n                        ),\n                    )\n                ),\n            )\n        )\n\n        self.plot_freq = int(\n            os.environ.get(\n                "RATER_PLOT_FREQ",\n                str(\n                    max(\n                        1,\n                        self.eval_freq,\n                    )\n                ),\n            )\n        )\n\n        # Save PNGs only. Diagnostic CSVs are disabled to conserve Drive.\n        self.save_plot_data = False\n\n        # ----------------------------------------------------\n        # Final weighted classifier hyperparameters.\n        # These have defaults, so config.py does not HAVE to\n        # define them unless you want command-line control.\n        # ----------------------------------------------------\n        self.final_epochs = int(\n            getattr(self.config, "rater_final_epochs", 50)\n        )\n        self.final_lr = float(\n            getattr(self.config, "rater_final_lr", 1e-2)\n        )\n        self.final_momentum = float(\n            getattr(self.config, "rater_final_momentum", 0.9)\n        )\n        self.final_weight_decay = float(\n            getattr(self.config, "rater_final_weight_decay", 1e-4)\n        )\n        # For robustness experiments we select the downstream\n        # classifier by held-out validation WGA by default.\n        #\n        # NOTE: WGA selection uses validation group labels. If you want a\n        # group-label-free selection rule, set rater_final_selection to\n        # "accuracy" or "loss" in config.py.\n        self.final_selection = str(\n            getattr(self.config, "rater_final_selection", "wga")\n        ).lower()\n\n        # Save final-classifier weight histograms every epoch by default.\n        # No config.py change is required.\n        self.final_plot_freq = int(\n            getattr(self.config, "rater_final_plot_freq", 1)\n        )\n\n        if self.inner_steps < 1:\n            raise ValueError("rater_inner_steps must be >= 1.")\n        if self.num_inner_models < 1:\n            raise ValueError("rater_num_inner_models must be >= 1.")\n        if self.temperature <= 0:\n            raise ValueError("rater_temperature must be > 0.")\n\n        if self.weighting not in {"softmax", "sigmoid"}:\n            raise ValueError(\n                "rater_weighting must be either \'softmax\' or \'sigmoid\'."\n            )\n\n        if self.final_selection not in {"accuracy", "wga", "loss"}:\n            raise ValueError(\n                "rater_final_selection must be one of: accuracy, wga, loss"\n            )\n\n        if self.final_plot_freq < 1:\n            raise ValueError("rater_final_plot_freq must be >= 1.")\n\n        # ----------------------------------------------------\n        # EA-style calibration protocol for the FINAL classifier\n        # ----------------------------------------------------\n        #\n        # With --split_val < 1, prepare_data() creates:\n        #\n        #   val_subset1 -> calibration set\n        #   val_subset2 -> held-out selection set\n        #\n        # This updated Rater uses:\n        #\n        #   * original train_no_aug for inner-model training\n        #   * val_subset1 as the Rater outer/calibration signal\n        #   * val_subset1 to train the fresh final classifier\n        #   * val_subset2 only for final-classifier checkpoint selection\n        #   * test only for final reporting\n        #\n        # This keeps val_subset2 untouched during Rater meta-training.\n        # ----------------------------------------------------\n        self.use_calibration_final = bool(\n            getattr(\n                self.config,\n                "rater_use_calibration_final",\n                True,\n            )\n        )\n\n        # ----------------------------------------------------\n        # Rater\n        # ----------------------------------------------------\n        if self.rater_capacity == "small":\n            self.rater = EmbeddingRaterSmall(self.feature_dim).to(self.device)\n        elif self.rater_capacity == "medium":\n            self.rater = EmbeddingRaterMedium(self.feature_dim).to(self.device)\n        else:\n            raise ValueError(\n                f"Unknown rater capacity: {self.rater_capacity}"\n            )\n\n        self.outer_optimizer = torch.optim.Adam(\n            self.rater.parameters(), lr=self.outer_lr\n        )\n\n        self.inner_models = []\n        self.final_classifier = None\n\n        log(\n            f"[Rater] EA-style Rater transform = "\n            f"{self.weighting}; temperature = {self.temperature}"\n        )\n        log(\n            "[Rater] EA-style gate: correct samples get weight=1; "\n            "only misclassified samples use the Rater rating."\n        )\n        log(\n            "[Rater] Inner objective: L_inner = sum_i w_i * CE_i "\n            "WITHOUT dividing by sum(w)."\n        )\n        log(\n            "[Rater] Meta objective = OUTER CLASSIFICATION "\n            "CROSS-ENTROPY ONLY."\n        )\n        log(\n            f"[Rater] Inner population = {self.num_inner_models}; "\n            "both inner classifiers are initialized independently from scratch."\n        )\n\n        if self.config.check_point:\n            self._load_rater_checkpoint(self.config.check_point)\n\n    # ========================================================\n    # Persistent shared embedding cache\n    # ========================================================\n\n    @staticmethod\n    def _safe_cache_token(value):\n        """Convert a value into a filesystem-safe cache token."""\n        token = str(value)\n        token = token.replace("/", "_")\n        token = token.replace(chr(92), "_")\n        token = token.replace(" ", "_")\n        token = token.replace(".", "p")\n        return token\n\n    def _embedding_cache_namespace(self):\n        """\n        Namespace shared embeddings by the EXACT frozen ERM representation.\n\n        This intentionally differs from the old ImageNet-only cache.\n        """\n        dataset_name = self._safe_cache_token(\n            getattr(\n                self.config,\n                "dataset",\n                "dataset",\n            )\n        )\n\n        backbone_name = self._safe_cache_token(\n            self.config.backbone\n        )\n\n        resolution = self._safe_cache_token(\n            getattr(\n                self.config,\n                "resolution",\n                224,\n            )\n        )\n\n        representation_tag = (\n            f"{backbone_name}_waterbirds_erm_"\n            f"{self.erm_checkpoint_fingerprint}"\n        )\n\n        return os.path.join(\n            dataset_name,\n            representation_tag,\n            f"resolution_{resolution}",\n        )\n\n    def _resolve_embedding_cache_dir(self, output_dir=None):\n        """\n        Return the embedding directory used by every train/test call.\n\n        If RATER_EMBEDDING_CACHE_ROOT is set, timestamped experiments all\n        share the same tensors. Otherwise fall back to the old per-run cache.\n        """\n        if self.embedding_cache_root:\n            cache_dir = os.path.join(\n                self.embedding_cache_root,\n                self._embedding_cache_namespace(),\n            )\n        else:\n            if output_dir is None:\n                raise ValueError(\n                    "No shared embedding cache root was configured and "\n                    "output_dir is None."\n                )\n\n            cache_dir = os.path.join(\n                output_dir,\n                "embedding_cache",\n            )\n\n        os.makedirs(cache_dir, exist_ok=True)\n        return cache_dir\n\n    def _embedding_cache_metadata(self, split):\n        """Metadata used to validate persistent cache files."""\n        metadata = {\n            "dataset": str(\n                getattr(self.config, "dataset", "")\n            ),\n            "split": str(split),\n            "backbone": str(self.config.backbone),\n            "pretrained": bool(self.config.pretrained),\n            "resolution": int(\n                getattr(self.config, "resolution", 224)\n            ),\n            "feature_dim": int(self.feature_dim),\n            "n_classes": int(self.n_classes),\n            "representation_source": (\n                "waterbirds_erm_checkpoint"\n            ),\n            "erm_checkpoint_path": os.path.abspath(\n                self.erm_model_path\n            ),\n            "erm_checkpoint_fingerprint": (\n                self.erm_checkpoint_fingerprint\n            ),\n            "erm_checkpoint_epoch": (\n                self.erm_checkpoint_epoch\n            ),\n        }\n\n        if "subset" in str(split):\n            metadata.update(\n                {\n                    "split_train": float(\n                        getattr(self.config, "split_train", 1.0)\n                    ),\n                    "split_val": float(\n                        getattr(self.config, "split_val", 1.0)\n                    ),\n                    "seed": int(\n                        getattr(self.config, "seed", 0)\n                    ),\n                }\n            )\n\n        return metadata\n\n    def _embedding_cache_filename(self, split):\n        """\n        Standard deterministic splits become:\n            train_no_aug.pt\n            val.pt\n            test.pt\n\n        Subset splits additionally encode split ratios and seed.\n        """\n        safe_split = self._safe_cache_token(split)\n\n        if "subset" in str(split):\n            split_train = self._safe_cache_token(\n                getattr(self.config, "split_train", 1.0)\n            )\n            split_val = self._safe_cache_token(\n                getattr(self.config, "split_val", 1.0)\n            )\n            seed = self._safe_cache_token(\n                getattr(self.config, "seed", 0)\n            )\n\n            safe_split = (\n                f"{safe_split}"\n                f"_splittrain_{split_train}"\n                f"_splitval_{split_val}"\n                f"_seed_{seed}"\n            )\n\n        return f"{safe_split}.pt"\n\n    @staticmethod\n    def _cache_payload_has_required_tensors(payload):\n        return (\n            isinstance(payload, dict)\n            and "embeddings" in payload\n            and "labels" in payload\n            and "groups" in payload\n            and "attrs" in payload\n        )\n\n    def _cache_metadata_matches(self, payload, split):\n        """Reject stale or incompatible shared cache files."""\n        if not self._cache_payload_has_required_tensors(payload):\n            return False\n\n        embeddings = payload["embeddings"]\n\n        if (\n            not torch.is_tensor(embeddings)\n            or embeddings.ndim != 2\n            or embeddings.shape[1] != self.feature_dim\n        ):\n            return False\n\n        expected = self._embedding_cache_metadata(split)\n        actual = payload.get("cache_metadata", None)\n\n        if actual is None:\n            return False\n\n        for key, expected_value in expected.items():\n            if actual.get(key) != expected_value:\n                return False\n\n        n = embeddings.shape[0]\n\n        return all(\n            torch.is_tensor(payload[key])\n            and payload[key].shape[0] == n\n            for key in ("labels", "groups", "attrs")\n        )\n\n    def _load_embedding_cache_file(self, cache_file, split):\n        """\n        Load one persistent cache file, returning None if it must be rebuilt.\n        """\n        if not os.path.exists(cache_file):\n            return None\n\n        log(\n            f"[Rater embeddings] Found shared cache: {cache_file}"\n        )\n\n        try:\n            payload = torch.load(\n                cache_file,\n                map_location="cpu",\n                weights_only=False,\n            )\n        except Exception as exc:\n            log(\n                f"[Rater embeddings] Cache load failed "\n                f"({type(exc).__name__}: {exc}). "\n                f"Regenerating split \'{split}\'."\n            )\n            return None\n\n        if not self._cache_metadata_matches(payload, split):\n            log(\n                f"[Rater embeddings] Cache metadata mismatch for "\n                f"split \'{split}\'. Regenerating it once."\n            )\n            return None\n\n        log(\n            f"[Rater embeddings] USING stored embeddings for "\n            f"\'{split}\' ({len(payload[\'labels\'])} samples). "\n            f"No backbone extraction is performed."\n        )\n\n        return EmbeddingTensorDataset(\n            payload["embeddings"],\n            payload["labels"],\n            payload["groups"],\n            payload["attrs"],\n        )\n\n    @torch.no_grad()\n    def _extract_embedding_dataset(self, split, cache_dir=None):\n        """\n        Load a split from persistent storage.\n\n        On the first ever request, extract with the frozen backbone once and\n        save it. Every later run directly loads the stored tensor file.\n        """\n        if cache_dir is None:\n            cache_dir = self._resolve_embedding_cache_dir()\n\n        os.makedirs(cache_dir, exist_ok=True)\n\n        cache_file = os.path.join(\n            cache_dir,\n            self._embedding_cache_filename(split),\n        )\n\n        cached_dataset = self._load_embedding_cache_file(\n            cache_file,\n            split,\n        )\n\n        if cached_dataset is not None:\n            return cached_dataset\n\n        if split not in self.dataloaders:\n            raise ValueError(\n                f"Unknown split \'{split}\'. Available splits: "\n                f"{list(self.dataloaders.keys())}"\n            )\n\n        log(\n            f"[Rater embeddings] Shared cache MISS for \'{split}\'. "\n            f"Extracting it ONCE with frozen "\n            f"{self.config.backbone}..."\n        )\n\n        loader = self.dataloaders[split]\n        self.feature_model.eval()\n\n        embeddings = []\n        labels = []\n        groups = []\n        attrs = []\n\n        for x, y, g, a in tqdm(\n            loader,\n            desc=f"ONE-TIME embedding extraction: {split}",\n        ):\n            x = x.to(\n                self.device,\n                non_blocking=True,\n            )\n\n            z = self.feature_model.backbone(x)\n\n            embeddings.append(z.detach().cpu())\n            labels.append(torch.as_tensor(y).cpu())\n            groups.append(torch.as_tensor(g).cpu())\n            attrs.append(torch.as_tensor(a).cpu())\n\n        embeddings = torch.cat(embeddings, dim=0)\n        labels = torch.cat(labels, dim=0)\n        groups = torch.cat(groups, dim=0)\n        attrs = torch.cat(attrs, dim=0)\n\n        payload = {\n            "embeddings": embeddings,\n            "labels": labels,\n            "groups": groups,\n            "attrs": attrs,\n            "cache_metadata": self._embedding_cache_metadata(split),\n        }\n\n        tmp_file = cache_file + f".tmp.{os.getpid()}"\n        torch.save(payload, tmp_file)\n        os.replace(tmp_file, cache_file)\n\n        log(\n            f"[Rater embeddings] SAVED persistent shared "\n            f"\'{split}\' embeddings ({len(labels)} samples):"\n        )\n        log(f"[Rater embeddings]   {cache_file}")\n        log(\n            "[Rater embeddings] Future runs will load this file "\n            "directly and will NOT extract this split again."\n        )\n\n        return EmbeddingTensorDataset(\n            embeddings,\n            labels,\n            groups,\n            attrs,\n        )\n\n    def _precompute_shared_embeddings(self, output_dir):\n        """\n        Build/validate all deterministic embeddings without meta-training.\n\n        Current Waterbirds workflow:\n            train_no_aug\n            val\n            test\n\n        Future subset/calibration splits are cached too when available.\n        """\n        cache_dir = self._resolve_embedding_cache_dir(output_dir)\n\n        preferred_splits = [\n            "train_no_aug",\n            "val",\n            "test",\n            "train_subset1",\n            "train_subset2",\n            "val_subset1",\n            "val_subset2",\n        ]\n\n        available = [\n            split\n            for split in preferred_splits\n            if split in self.dataloaders\n        ]\n\n        log("[Rater embeddings] PRECOMPUTE-ONLY mode.")\n        log(\n            f"[Rater embeddings] Shared cache directory: {cache_dir}"\n        )\n        log(\n            f"[Rater embeddings] Preparing splits: {available}"\n        )\n\n        for split in available:\n            dataset = self._extract_embedding_dataset(\n                split,\n                cache_dir,\n            )\n            log(\n                f"[Rater embeddings] READY: "\n                f"{split} -> {len(dataset)} samples"\n            )\n\n        ready_file = os.path.join(cache_dir, "_READY.txt")\n\n        with open(ready_file, "w") as f:\n            f.write("Persistent Rater embedding cache is ready.\\n")\n            for split in available:\n                f.write(\n                    self._embedding_cache_filename(split)\n                    + "\\n"\n                )\n\n        log(\n            f"[Rater embeddings] Cache preparation complete: "\n            f"{ready_file}"\n        )\n\n    # ========================================================\n    # Inner models\n    # ========================================================\n\n    def _new_inner_model(\n        self,\n        model_index=0,\n    ):\n        """\n        Fresh random Linear(feature_dim, n_classes) classifier.\n\n        IMPORTANT: the saved ERM classifier head is NOT used here.\n        The ERM checkpoint is used only to define the frozen backbone\n        representation / cached embeddings.\n        """\n        del model_index\n\n        return InnerLinearClassifier(\n            input_dim=self.feature_dim,\n            num_classes=self.n_classes,\n        ).to(self.device)\n\n    def _new_final_classifier(self):\n        # Kept only for API compatibility; this meta-only experiment never\n        # trains a final classifier.\n        return InnerLinearClassifier(\n            input_dim=self.feature_dim,\n            num_classes=self.n_classes,\n        ).to(self.device)\n\n    def _initialize_inner_population(self):\n        if self.num_inner_models != 2:\n            raise ValueError(\n                "This experiment is defined with exactly two inner models."\n            )\n\n        self.inner_models = [\n            self._new_inner_model(model_index=i)\n            for i in range(2)\n        ]\n\n        with torch.no_grad():\n            dw = (\n                self.inner_models[0].linear.weight\n                - self.inner_models[1].linear.weight\n            ).norm().item()\n\n            db = (\n                self.inner_models[0].linear.bias\n                - self.inner_models[1].linear.bias\n            ).norm().item()\n\n        if dw == 0.0 and db == 0.0:\n            raise RuntimeError(\n                "The two independently-created inner classifiers are identical."\n            )\n\n        log(\n            f"[Rater inner init] two independent RANDOM classifiers; "\n            f"||dW||={dw:.6f}, ||db||={db:.6f}. "\n            "ERM classifier initialization is NOT used."\n        )\n\n    @staticmethod\n    def _next_batch(iterator, loader):\n        try:\n            batch = next(iterator)\n        except StopIteration:\n            iterator = iter(loader)\n            batch = next(iterator)\n        return batch, iterator\n\n    # ========================================================\n    # Raw Rater score -> rating transformation\n    # ========================================================\n\n    def _scores_to_weights(self, raw_scores):\n        """\n        Convert raw Rater outputs into ratings in (0, 1).\n\n        In the EA-style rule this function is applied to MISCLASSIFIED\n        samples only. Correct samples receive weight exactly 1.\n        """\n\n        if raw_scores.numel() == 0:\n            return raw_scores\n\n        if self.weighting == "softmax":\n            centered_scores = raw_scores - raw_scores.mean()\n            return torch.softmax(\n                centered_scores / self.temperature,\n                dim=0,\n            )\n\n        score_mean = raw_scores.mean()\n        score_std = raw_scores.std(unbiased=False)\n        standardized_scores = (\n            raw_scores - score_mean\n        ) / (\n            score_std + 1e-6\n        )\n        return torch.sigmoid(\n            standardized_scores / self.temperature\n        )\n\n    def _all_sample_training_weights(\n        self,\n        z,\n        y,\n        logits,\n        raw_scores_all=None,\n    ):\n        """\n        Rate EVERY sample.\n\n        Correct examples are NOT forced to weight 1.\n\n            raw_i = Rater(z_i)\n            w_i   = transform(raw_i)\n\n        `correct_mask` is returned only for diagnostics.\n        """\n        if raw_scores_all is None:\n            raw_scores = self.rater(z)\n        else:\n            raw_scores = raw_scores_all\n\n        weights = self._scores_to_weights(\n            raw_scores\n        )\n\n        correct_mask = (\n            logits.argmax(dim=1).eq(y)\n        )\n\n        return (\n            weights,\n            correct_mask,\n            raw_scores,\n        )\n\n\n    def _ea_style_training_weights(\n        self,\n        z,\n        y,\n        logits,\n        raw_scores_all=None,\n    ):\n        """\n        EA-style training weights from the supplied implementation:\n\n            correct prediction -> weight 1\n            wrong prediction   -> transformed Rater rating\n\n        The Rater is evaluated only on misclassified samples during\n        actual inner training. For diagnostics, raw_scores_all may be\n        supplied and then indexed on the misclassified subset.\n        """\n        pred = logits.argmax(dim=1)\n        correct_mask = pred.eq(y)\n        misclassified_mask = ~correct_mask\n\n        weights = torch.ones(\n            y.shape[0],\n            device=logits.device,\n            dtype=logits.dtype,\n        )\n\n        if not bool(misclassified_mask.any()):\n            empty_scores = torch.empty(\n                0,\n                device=logits.device,\n                dtype=logits.dtype,\n            )\n            return weights, correct_mask, empty_scores\n\n        misclassified_indices = torch.nonzero(\n            misclassified_mask,\n            as_tuple=False,\n        ).squeeze(1)\n\n        if raw_scores_all is None:\n            misclassified_raw_scores = self.rater(\n                z[misclassified_mask]\n            )\n        else:\n            misclassified_raw_scores = raw_scores_all[\n                misclassified_mask\n            ]\n\n        misclassified_ratings = self._scores_to_weights(\n            misclassified_raw_scores\n        )\n\n        weights = weights.index_copy(\n            0,\n            misclassified_indices,\n            misclassified_ratings,\n        )\n\n        return (\n            weights,\n            correct_mask,\n            misclassified_raw_scores,\n        )\n\n    # ========================================================\n    # Differentiable inner optimization\n    # ========================================================\n\n    @staticmethod\n    def _within_class_percentile_targets(\n        values,\n        labels,\n    ):\n        """\n        Percentile-rank scalar values independently within each class.\n\n        Output:\n            target in [0, 1]\n\n        If only one example of a class is present in a minibatch,\n        that sample gets the neutral target 0.5.\n        """\n        with torch.no_grad():\n            targets = torch.empty_like(\n                values,\n                dtype=torch.float32,\n            )\n\n            for class_id in torch.unique(\n                labels\n            ):\n                mask = labels.eq(\n                    class_id\n                )\n\n                class_values = values[\n                    mask\n                ]\n\n                n = class_values.numel()\n\n                if n == 1:\n                    class_targets = torch.full_like(\n                        class_values,\n                        0.5,\n                        dtype=torch.float32,\n                    )\n                else:\n                    order = torch.argsort(\n                        class_values,\n                        stable=True,\n                    )\n\n                    ranks = torch.empty(\n                        n,\n                        device=values.device,\n                        dtype=torch.float32,\n                    )\n\n                    ranks[\n                        order\n                    ] = torch.linspace(\n                        0.0,\n                        1.0,\n                        steps=n,\n                        device=values.device,\n                        dtype=torch.float32,\n                    )\n\n                    class_targets = ranks\n\n                targets[\n                    mask\n                ] = class_targets\n\n            return targets\n\n\n    @torch.no_grad()\n    def _per_sample_param_grad_norm(\n        self,\n        z,\n        y,\n        logits,\n    ):\n        """\n        Exact per-sample parameter-gradient magnitude of the CURRENT\n        linear classifier.\n\n        logits_i = W z_i + b\n\n        delta_i = softmax(logits_i) - onehot(y_i)\n\n        dL_i/dW = delta_i outer z_i\n        dL_i/db = delta_i\n\n        Thus:\n\n            ||dL_i/d(W,b)||_2\n                = ||delta_i||_2 * sqrt(||z_i||_2^2 + 1)\n        """\n        probs = torch.softmax(\n            logits.detach(),\n            dim=1,\n        )\n\n        onehot = F.one_hot(\n            y,\n            num_classes=self.n_classes,\n        ).to(\n            probs.dtype\n        )\n\n        delta = (\n            probs - onehot\n        )\n\n        delta_norm = torch.linalg.vector_norm(\n            delta,\n            ord=2,\n            dim=1,\n        )\n\n        z_norm_sq = (\n            z.detach()\n            .pow(2)\n            .sum(dim=1)\n        )\n\n        return (\n            delta_norm\n            * torch.sqrt(\n                z_norm_sq + 1.0\n            )\n        )\n\n\n    def _gradient_percentile_mse(\n        self,\n        raw_scores,\n        z,\n        y,\n        logits,\n    ):\n        """\n        Auxiliary Rater supervision.\n\n            G_i = exact per-sample parameter-gradient norm\n            q_i = within-class percentile of G_i\n            r_i = sigmoid(raw_rater_score_i)\n\n            L_grad = mean_i (r_i - q_i)^2\n\n        ALL samples contribute.\n        Group/background labels are never used.\n        """\n        grad_norm = (\n            self._per_sample_param_grad_norm(\n                z=z,\n                y=y,\n                logits=logits,\n            )\n        )\n\n        target = (\n            self._within_class_percentile_targets(\n                grad_norm,\n                y,\n            )\n        )\n\n        prediction = torch.sigmoid(\n            raw_scores\n        )\n\n        grad_mse = F.mse_loss(\n            prediction,\n            target,\n            reduction="mean",\n        )\n\n        return (\n            grad_mse,\n            grad_norm.detach(),\n            target.detach(),\n            prediction,\n        )\n\n\n    def _inner_unroll(\n        self,\n        inner_model,\n        train_iterator,\n        train_loader,\n    ):\n        """\n        Differentiable FULL-EPOCH EA-style weighted classifier optimization.\n\n        On each batch:\n            correct -> weight 1\n            wrong   -> Rater rating\n\n        Inner objective:\n            L_inner = SUM_i weight_i * CE_i\n\n        There is NO division by sum(weights).\n        """\n        fast_weight = (\n            inner_model.linear.weight\n            .detach()\n            .clone()\n            .requires_grad_(True)\n        )\n\n        fast_bias = (\n            inner_model.linear.bias\n            .detach()\n            .clone()\n            .requires_grad_(True)\n        )\n\n        last_raw_scores = None\n\n        for _ in range(self.inner_steps):\n            batch, train_iterator = self._next_batch(\n                train_iterator,\n                train_loader,\n            )\n\n            z, y, _, _ = batch\n            z = z.to(self.device, non_blocking=True)\n            y = y.to(self.device, non_blocking=True)\n\n            logits = F.linear(\n                z,\n                fast_weight,\n                fast_bias,\n            )\n\n            per_sample_loss = F.cross_entropy(\n                logits,\n                y,\n                reduction="none",\n            )\n\n            weights, _, misclassified_raw_scores = (\n                self._ea_style_training_weights(\n                    z=z,\n                    y=y,\n                    logits=logits,\n                    raw_scores_all=None,\n                )\n            )\n\n            inner_loss = (\n                per_sample_loss * weights\n            ).sum()\n\n            if self.inner_reg_weight > 0:\n                inner_loss = (\n                    inner_loss\n                    + self.inner_reg_weight\n                    * (\n                        fast_weight.pow(2).sum()\n                        + fast_bias.pow(2).sum()\n                    )\n                )\n\n            grad_w, grad_b = torch.autograd.grad(\n                inner_loss,\n                [fast_weight, fast_bias],\n                create_graph=True,\n            )\n\n            fast_weight = (\n                fast_weight\n                - self.inner_lr * grad_w\n            )\n            fast_bias = (\n                fast_bias\n                - self.inner_lr * grad_b\n            )\n\n            last_raw_scores = misclassified_raw_scores\n\n        return (\n            {\n                "weight": fast_weight,\n                "bias": fast_bias,\n            },\n            train_iterator,\n            last_raw_scores,\n            torch.zeros((), device=self.device),\n        )\n\n    # ========================================================\n    # One bilevel/meta step\n    # ========================================================\n\n    def _meta_step(\n        self,\n        train_iterator,\n        val_iterator,\n        train_loader,\n        val_loader,\n    ):\n        """\n        One classification-only Rater update.\n\n        BOTH persistent inner models complete self.inner_steps batches\n        before the single optimizer.step() on the Rater. Since train() sets\n        self.inner_steps = len(train_loader), both models see the entire\n        training set once before every Rater update.\n\n        For memory efficiency, each model\'s outer loss / num_models is\n        backpropagated immediately, accumulating Rater gradients. The Rater\n        optimizer step happens only after BOTH models are finished.\n        """\n        self.rater.train()\n\n        outer_batch, val_iterator = self._next_batch(\n            val_iterator,\n            val_loader,\n        )\n\n        z_outer, y_outer, _, _ = outer_batch\n        z_outer = z_outer.to(self.device, non_blocking=True)\n        y_outer = y_outer.to(self.device, non_blocking=True)\n\n        self.outer_optimizer.zero_grad()\n\n        outer_values = []\n        committed_fast_params = []\n\n        for inner_model in self.inner_models:\n            (\n                fast_params,\n                train_iterator,\n                _,\n                _,\n            ) = self._inner_unroll(\n                inner_model,\n                train_iterator,\n                train_loader,\n            )\n\n            outer_logits = F.linear(\n                z_outer,\n                fast_params["weight"],\n                fast_params["bias"],\n            )\n\n            outer_loss = F.cross_entropy(\n                outer_logits,\n                y_outer,\n            )\n\n            (\n                outer_loss / self.num_inner_models\n            ).backward()\n\n            outer_values.append(\n                float(outer_loss.detach().item())\n            )\n\n            committed_fast_params.append(\n                {\n                    "weight": fast_params["weight"].detach(),\n                    "bias": fast_params["bias"].detach(),\n                }\n            )\n\n        if self.grad_clip > 0:\n            torch.nn.utils.clip_grad_norm_(\n                self.rater.parameters(),\n                self.grad_clip,\n            )\n\n        self.outer_optimizer.step()\n\n        with torch.no_grad():\n            for inner_model, fast_params in zip(\n                self.inner_models,\n                committed_fast_params,\n            ):\n                inner_model.linear.weight.copy_(\n                    fast_params["weight"]\n                )\n                inner_model.linear.bias.copy_(\n                    fast_params["bias"]\n                )\n\n        outer_ce = float(np.mean(outer_values))\n\n        return (\n            outer_ce,\n            outer_ce,\n            0.0,\n            train_iterator,\n            val_iterator,\n        )\n\n    # ========================================================\n    # Evaluation helpers\n    # ========================================================\n\n    @torch.no_grad()\n    def _evaluate_classifier(self, model, dataset):\n        """\n        Evaluate an embedding classifier.\n\n        Returns standard loss/accuracy plus Waterbirds-style per-group\n        accuracy and worst-group accuracy.\n        """\n\n        loader = DataLoader(\n            dataset,\n            batch_size=self.config.batch_size,\n            shuffle=False,\n            num_workers=self.config.num_workers,\n            pin_memory=True,\n        )\n\n        model.eval()\n\n        total_loss = 0.0\n        total_correct = 0\n        total_examples = 0\n        group_correct = {}\n        group_total = {}\n\n        for z, y, g, _ in loader:\n            z = z.to(self.device, non_blocking=True)\n            y = y.to(self.device, non_blocking=True)\n            g = g.to(self.device, non_blocking=True)\n\n            logits = model(z)\n            loss = F.cross_entropy(\n                logits,\n                y,\n                reduction="sum",\n            )\n            pred = logits.argmax(dim=1)\n\n            total_loss += loss.item()\n            total_correct += (pred == y).sum().item()\n            total_examples += y.numel()\n\n            for group_id in torch.unique(g):\n                gid = int(group_id.item())\n                mask = g == group_id\n                group_correct[gid] = (\n                    group_correct.get(gid, 0)\n                    + (pred[mask] == y[mask]).sum().item()\n                )\n                group_total[gid] = (\n                    group_total.get(gid, 0)\n                    + mask.sum().item()\n                )\n\n        if total_examples == 0:\n            raise RuntimeError("Cannot evaluate an empty dataset.")\n\n        group_accuracy = {\n            gid: group_correct[gid] / group_total[gid]\n            for gid in sorted(group_total)\n            if group_total[gid] > 0\n        }\n\n        return {\n            "loss": total_loss / total_examples,\n            "accuracy": total_correct / total_examples,\n            "group_accuracy": group_accuracy,\n            "worst_group_accuracy": min(group_accuracy.values())\n            if group_accuracy else float("nan"),\n        }\n\n    def _evaluate_inner_population(self, dataset, verbose=True):\n        """Evaluate every persistent inner model and aggregate metrics."""\n\n        metrics = []\n\n        for i, model in enumerate(self.inner_models):\n            result = self._evaluate_classifier(model, dataset)\n            metrics.append(result)\n\n            if verbose:\n                group_str = ", ".join(\n                    f"g{gid}={100.0 * acc:.2f}%"\n                    for gid, acc in result["group_accuracy"].items()\n                )\n                log(\n                    f"[Inner model {i}] "\n                    f"loss={result[\'loss\']:.6f}, "\n                    f"acc={100.0 * result[\'accuracy\']:.2f}%, "\n                    f"WGA={100.0 * result[\'worst_group_accuracy\']:.2f}% "\n                    f"({group_str})"\n                )\n\n        mean_acc = float(np.mean([m["accuracy"] for m in metrics]))\n        mean_wga = float(\n            np.mean([m["worst_group_accuracy"] for m in metrics])\n        )\n        mean_loss = float(np.mean([m["loss"] for m in metrics]))\n        best_acc = float(max(m["accuracy"] for m in metrics))\n        best_wga = float(\n            max(m["worst_group_accuracy"] for m in metrics)\n        )\n\n        if verbose:\n            log(\n                f"[Inner population] mean_loss={mean_loss:.6f}, "\n                f"mean_acc={100.0 * mean_acc:.2f}%, "\n                f"mean_WGA={100.0 * mean_wga:.2f}%, "\n                f"best_acc={100.0 * best_acc:.2f}%, "\n                f"best_WGA={100.0 * best_wga:.2f}%"\n            )\n\n        return {\n            "models": metrics,\n            "mean_loss": mean_loss,\n            "mean_accuracy": mean_acc,\n            "mean_wga": mean_wga,\n            "best_accuracy": best_acc,\n            "best_wga": best_wga,\n        }\n\n    @torch.no_grad()\n    def _score_loss_relationship(self, dataset, models=None):\n        """\n        Diagnostic relationship between raw Rater scores, effective\n        EA-style weights, classifier loss, and correctness.\n\n        For each inner model:\n            correct -> weight 1\n            wrong   -> Rater rating\n\n        `final_scores` is the mean effective weight across the current\n        inner-model population.\n        """\n\n        if models is None:\n            models = self.inner_models\n\n        loader = DataLoader(\n            dataset,\n            batch_size=self.config.batch_size,\n            shuffle=False,\n            num_workers=self.config.num_workers,\n            pin_memory=True,\n        )\n\n        self.rater.eval()\n        for model in models:\n            model.eval()\n\n        all_scores = []\n        all_final_scores = []\n        all_losses = []\n        all_correct = []\n        all_groups = []\n        all_labels = []\n        all_attrs = []\n\n        for z, y, g, a in loader:\n            z = z.to(self.device, non_blocking=True)\n            y_device = y.to(self.device, non_blocking=True)\n\n            # Raw scores for all samples are used only for diagnostics.\n            raw_scores = self.rater(z)\n\n            model_losses = []\n            model_correct = []\n            model_weights = []\n\n            for model in models:\n                logits = model(z)\n                per_loss = F.cross_entropy(\n                    logits,\n                    y_device,\n                    reduction="none",\n                )\n\n                effective_weights, correct_mask, _ = (\n                    self._all_sample_training_weights(\n                        z=z,\n                        y=y_device,\n                        logits=logits,\n                        raw_scores_all=raw_scores,\n                    )\n                )\n\n                model_losses.append(per_loss)\n                model_correct.append(correct_mask.float())\n                model_weights.append(effective_weights)\n\n            mean_loss = torch.stack(model_losses, dim=0).mean(dim=0)\n            mean_correct = torch.stack(model_correct, dim=0).mean(dim=0)\n            mean_weight = torch.stack(model_weights, dim=0).mean(dim=0)\n\n            all_scores.append(raw_scores.cpu())\n            all_final_scores.append(mean_weight.cpu())\n            all_losses.append(mean_loss.cpu())\n            all_correct.append(mean_correct.cpu())\n            all_groups.append(g.cpu())\n            all_labels.append(y.cpu())\n            all_attrs.append(a.cpu())\n\n        scores = torch.cat(all_scores).numpy()\n        final_scores = torch.cat(all_final_scores).numpy()\n        losses = torch.cat(all_losses).numpy()\n        correctness = torch.cat(all_correct).numpy()\n        groups = torch.cat(all_groups).numpy()\n        labels = torch.cat(all_labels).numpy()\n        attrs = torch.cat(all_attrs).numpy()\n\n        def _corr(x, y):\n            if len(x) >= 2 and np.std(x) > 0 and np.std(y) > 0:\n                return (\n                    float(stats.spearmanr(x, y).statistic),\n                    float(stats.pearsonr(x, y).statistic),\n                )\n            return 0.0, 0.0\n\n        spearman_loss, pearson_loss = _corr(scores, losses)\n        spearman_final_loss, pearson_final_loss = _corr(\n            final_scores, losses\n        )\n\n        if len(scores) >= 2 and np.std(scores) > 0 and np.std(correctness) > 0:\n            spearman_correct = float(\n                stats.spearmanr(scores, correctness).statistic\n            )\n        else:\n            spearman_correct = 0.0\n\n        if len(final_scores) >= 2 and np.std(final_scores) > 0 and np.std(correctness) > 0:\n            spearman_final_correct = float(\n                stats.spearmanr(final_scores, correctness).statistic\n            )\n        else:\n            spearman_final_correct = 0.0\n\n        return {\n            "scores": scores,\n            "final_scores": final_scores,\n            "training_weights": final_scores,\n            "mean_loss": losses,\n            "mean_correctness": correctness,\n            "groups": groups,\n            "labels": labels,\n            "attrs": attrs,\n            "spearman_score_vs_loss": spearman_loss,\n            "pearson_score_vs_loss": pearson_loss,\n            "spearman_score_vs_correctness": spearman_correct,\n            "spearman_final_score_vs_loss": spearman_final_loss,\n            "pearson_final_score_vs_loss": pearson_final_loss,\n            "spearman_final_score_vs_correctness": spearman_final_correct,\n        }\n\n    def _group_display_name(self, gid):\n        """Readable group labels for Waterbirds; generic labels otherwise."""\n\n        if self.config.dataset == "waterbirds":\n            names = {\n                0: "Group 0: landbird / land",\n                1: "Group 1: landbird / water",\n                2: "Group 2: waterbird / land",\n                3: "Group 3: waterbird / water",\n            }\n            return names.get(int(gid), f"Group {int(gid)}")\n\n        return f"Group {int(gid)}"\n\n    def _save_rate_vs_loss_plot(\n        self,\n        relationship,\n        output_dir,\n        meta_step,\n        split_name="val",\n        tag_prefix="meta_step",\n    ):\n        """\n        Save a group-colored scatter plot of raw rater score vs mean\n        per-example inner-model CE loss.\n\n        Each point is one embedding. Each Waterbirds group is drawn as a\n        separate scatter series, so matplotlib automatically assigns a\n        different color to each group.\n        """\n\n        scores = np.asarray(relationship["scores"])\n        final_scores = np.asarray(relationship["final_scores"])\n        losses = np.asarray(relationship["mean_loss"])\n        groups = np.asarray(relationship["groups"])\n        labels = np.asarray(relationship["labels"])\n        attrs = np.asarray(relationship["attrs"])\n        correctness = np.asarray(\n            relationship["mean_correctness"]\n        )\n\n        plot_dir = os.path.join(\n            output_dir,\n            "plots",\n            "rate_vs_loss",\n        )\n        data_dir = os.path.join(\n            output_dir,\n            "plots",\n            "rate_vs_loss_data",\n        )\n\n        os.makedirs(plot_dir, exist_ok=True)\n        os.makedirs(data_dir, exist_ok=True)\n\n        fig, ax = plt.subplots(figsize=(10, 7))\n\n        for gid in sorted(np.unique(groups)):\n            mask = groups == gid\n            ax.scatter(\n                losses[mask],\n                scores[mask],\n                alpha=0.50,\n                s=24,\n                label=self._group_display_name(gid),\n            )\n\n        # Overall least-squares line, matching the spirit of the user\'s\n        # previous evaluation script.\n        if (\n            len(losses) >= 2\n            and np.std(losses) > 0\n            and np.std(scores) > 0\n        ):\n            slope, intercept, r_value, _, _ = stats.linregress(\n                losses,\n                scores,\n            )\n            x_line = np.linspace(\n                float(losses.min()),\n                float(losses.max()),\n                200,\n            )\n            y_line = slope * x_line + intercept\n            ax.plot(\n                x_line,\n                y_line,\n                linewidth=2,\n                label=f"Overall linear fit (R²={r_value ** 2:.3f})",\n            )\n\n        ax.set_xlabel(\n            "Mean per-example cross-entropy loss across inner models"\n        )\n        ax.set_ylabel("Raw rater score")\n        ax.set_title(\n            f"Rater score vs inner-model loss | {split_name} | "\n            f"step {meta_step}\\n"\n            f"Spearman={relationship[\'spearman_score_vs_loss\']:.3f}, "\n            f"Pearson={relationship[\'pearson_score_vs_loss\']:.3f}"\n        )\n        ax.grid(True, linestyle=":", alpha=0.35)\n        ax.legend(loc="best", fontsize=9)\n        fig.tight_layout()\n\n        plot_path = os.path.join(\n            plot_dir,\n            f"{tag_prefix}_{meta_step:06d}_{split_name}.png",\n        )\n        fig.savefig(\n            plot_path,\n            dpi=160,\n            bbox_inches="tight",\n        )\n        plt.close(fig)\n\n        if self.save_plot_data:\n            csv_path = os.path.join(\n                data_dir,\n                f"{tag_prefix}_{meta_step:06d}_{split_name}.csv",\n            )\n\n            with open(csv_path, "w", newline="") as f:\n                writer = csv.writer(f)\n                writer.writerow(\n                    [\n                        "index",\n                        "raw_rater_score",\n                        "effective_ea_style_weight",\n                        "mean_inner_loss",\n                        "mean_inner_correctness",\n                        "group",\n                        "label",\n                        "attribute",\n                        "weighting_method",\n                        "temperature",\n                    ]\n                )\n\n                for i in range(len(scores)):\n                    writer.writerow(\n                        [\n                            i,\n                            float(scores[i]),\n                            float(final_scores[i]),\n                            float(losses[i]),\n                            float(correctness[i]),\n                            int(groups[i]),\n                            int(labels[i]),\n                            int(attrs[i]),\n                            self.weighting,\n                            float(self.temperature),\n                        ]\n                    )\n\n        log(\n            f"[Rater plot] saved score-vs-loss plot: {plot_path}"\n        )\n\n        return plot_path\n\n    def _save_group_score_histogram(\n        self,\n        relationship,\n        output_dir,\n        meta_step,\n        split_name="val",\n        tag_prefix="meta_step",\n        score_kind="raw",\n    ):\n        """\n        Save the distribution of Rater outputs separately for every group.\n\n        score_kind="raw":\n            histogram of unrestricted rater outputs r_eta(z).\n\n        score_kind="final":\n            histogram of the EFFECTIVE EA-STYLE WEIGHT actually used:\n            correct samples have weight 1; misclassified samples use the\n            selected Rater rating transform.\n\n        Density normalization is used so strongly imbalanced Waterbirds\n        groups can be compared by distribution shape rather than count.\n        """\n\n        if score_kind == "raw":\n            values = np.asarray(relationship["scores"])\n            folder = "raw_score_histogram"\n            xlabel = "Raw rater score"\n            title_name = "Raw Rater score distribution"\n            filename_kind = "raw_score_hist"\n        elif score_kind == "final":\n            values = np.asarray(relationship["final_scores"])\n            folder = f"final_score_histogram_{self.weighting}"\n            xlabel = (\n                "Effective EA-GATED Rater weight "\n                "(every sample=Rater weight)"\n            )\n            title_name = (\n                f"EA-GATED Rater weight distribution "\n                f"(wrong uses {self.weighting})"\n            )\n            filename_kind = "final_score_hist"\n        else:\n            raise ValueError(\n                f"Unknown score_kind={score_kind}; expected \'raw\' or \'final\'."\n            )\n\n        groups = np.asarray(relationship["groups"])\n\n        plot_dir = os.path.join(\n            output_dir,\n            "plots",\n            folder,\n        )\n        os.makedirs(plot_dir, exist_ok=True)\n\n        fig, ax = plt.subplots(figsize=(10, 7))\n\n        if len(values) == 0:\n            plt.close(fig)\n            return None\n\n        vmin = float(np.min(values))\n        vmax = float(np.max(values))\n        if np.isclose(vmin, vmax):\n            eps = max(abs(vmin) * 0.05, 1e-6)\n            bins = np.linspace(vmin - eps, vmax + eps, 20)\n        else:\n            bins = np.linspace(vmin, vmax, 41)\n\n        for gid in sorted(np.unique(groups)):\n            mask = groups == gid\n            group_values = values[mask]\n            if len(group_values) == 0:\n                continue\n\n            ax.hist(\n                group_values,\n                bins=bins,\n                density=True,\n                alpha=0.45,\n                label=(\n                    f"{self._group_display_name(gid)} "\n                    f"(n={int(mask.sum())})"\n                ),\n            )\n\n        ax.set_xlabel(xlabel)\n        ax.set_ylabel("Density")\n        ax.set_title(\n            f"{title_name} by group | {split_name} | step {meta_step}"\n        )\n        ax.grid(True, linestyle=":", alpha=0.30)\n        ax.legend(loc="best", fontsize=9)\n        fig.tight_layout()\n\n        plot_path = os.path.join(\n            plot_dir,\n            f"{tag_prefix}_{meta_step:06d}_{split_name}_{filename_kind}.png",\n        )\n        fig.savefig(\n            plot_path,\n            dpi=160,\n            bbox_inches="tight",\n        )\n        plt.close(fig)\n\n        log(\n            f"[Rater plot] saved {score_kind}-score histogram: "\n            f"{plot_path}"\n        )\n        return plot_path\n\n    def _save_final_score_vs_loss_plot(\n        self,\n        relationship,\n        output_dir,\n        meta_step,\n        split_name="val",\n        tag_prefix="meta_step",\n    ):\n        """\n        Save group-colored scatter of FINAL SCORE vs classifier loss.\n\n        FINAL SCORE means the effective EA-GATED Rater weight:\n        every sample -> Rater weight.\n        """\n\n        final_scores = np.asarray(relationship["final_scores"])\n        losses = np.asarray(relationship["mean_loss"])\n        groups = np.asarray(relationship["groups"])\n\n        plot_dir = os.path.join(\n            output_dir,\n            "plots",\n            f"final_score_vs_loss_{self.weighting}",\n        )\n        os.makedirs(plot_dir, exist_ok=True)\n\n        fig, ax = plt.subplots(figsize=(10, 7))\n\n        for gid in sorted(np.unique(groups)):\n            mask = groups == gid\n            ax.scatter(\n                losses[mask],\n                final_scores[mask],\n                alpha=0.50,\n                s=24,\n                label=self._group_display_name(gid),\n            )\n\n        if (\n            len(losses) >= 2\n            and np.std(losses) > 0\n            and np.std(final_scores) > 0\n        ):\n            slope, intercept, r_value, _, _ = stats.linregress(\n                losses,\n                final_scores,\n            )\n            x_line = np.linspace(\n                float(losses.min()),\n                float(losses.max()),\n                200,\n            )\n            y_line = slope * x_line + intercept\n            ax.plot(\n                x_line,\n                y_line,\n                linewidth=2,\n                label=f"Overall linear fit (R²={r_value ** 2:.3f})",\n            )\n\n        ax.set_xlabel(\n            "Mean per-example cross-entropy loss across inner models"\n        )\n        ax.set_ylabel(\n            "Effective EA-style weight (every sample=Rater weight)"\n        )\n        ax.set_title(\n            f"EA-GATED Rater weight vs inner-model loss "\n            f"(wrong uses {self.weighting}) | "\n            f"{split_name} | "\n            f"step {meta_step}\\n"\n            f"Spearman={relationship[\'spearman_final_score_vs_loss\']:.3f}, "\n            f"Pearson={relationship[\'pearson_final_score_vs_loss\']:.3f}"\n        )\n        ax.grid(True, linestyle=":", alpha=0.35)\n        ax.legend(loc="best", fontsize=9)\n        fig.tight_layout()\n\n        plot_path = os.path.join(\n            plot_dir,\n            f"{tag_prefix}_{meta_step:06d}_{split_name}.png",\n        )\n        fig.savefig(\n            plot_path,\n            dpi=160,\n            bbox_inches="tight",\n        )\n        plt.close(fig)\n\n        log(\n            f"[Rater plot] saved final-score-vs-loss plot: {plot_path}"\n        )\n        return plot_path\n\n    def _save_all_score_diagnostics(\n        self,\n        relationship,\n        output_dir,\n        meta_step,\n        split_name="val",\n        tag_prefix="meta_step",\n    ):\n        """Save all score diagnostics for one evaluation step."""\n\n        self._save_rate_vs_loss_plot(\n            relationship=relationship,\n            output_dir=output_dir,\n            meta_step=meta_step,\n            split_name=split_name,\n            tag_prefix=tag_prefix,\n        )\n        self._save_group_score_histogram(\n            relationship=relationship,\n            output_dir=output_dir,\n            meta_step=meta_step,\n            split_name=split_name,\n            tag_prefix=tag_prefix,\n            score_kind="raw",\n        )\n        self._save_group_score_histogram(\n            relationship=relationship,\n            output_dir=output_dir,\n            meta_step=meta_step,\n            split_name=split_name,\n            tag_prefix=tag_prefix,\n            score_kind="final",\n        )\n        self._save_final_score_vs_loss_plot(\n            relationship=relationship,\n            output_dir=output_dir,\n            meta_step=meta_step,\n            split_name=split_name,\n            tag_prefix=tag_prefix,\n        )\n\n    @torch.no_grad()\n    def _classifier_score_loss_relationship(self, dataset, model):\n        """Raw score and effective EA-style weight vs loss for one model."""\n\n        loader = DataLoader(\n            dataset,\n            batch_size=self.config.batch_size,\n            shuffle=False,\n            num_workers=self.config.num_workers,\n            pin_memory=True,\n        )\n\n        self.rater.eval()\n        model.eval()\n\n        all_scores = []\n        all_final_scores = []\n        all_losses = []\n        all_correct = []\n        all_groups = []\n        all_labels = []\n        all_attrs = []\n\n        for z, y, g, a in loader:\n            z = z.to(self.device, non_blocking=True)\n            y_device = y.to(self.device, non_blocking=True)\n\n            raw_scores = self.rater(z)\n            logits = model(z)\n\n            effective_weights, correct_mask, _ = (\n                self._all_sample_training_weights(\n                    z=z,\n                    y=y_device,\n                    logits=logits,\n                    raw_scores_all=raw_scores,\n                )\n            )\n\n            losses = F.cross_entropy(\n                logits,\n                y_device,\n                reduction="none",\n            )\n\n            all_scores.append(raw_scores.cpu())\n            all_final_scores.append(effective_weights.cpu())\n            all_losses.append(losses.cpu())\n            all_correct.append(correct_mask.float().cpu())\n            all_groups.append(g.cpu())\n            all_labels.append(y.cpu())\n            all_attrs.append(a.cpu())\n\n        scores = torch.cat(all_scores).numpy()\n        final_scores = torch.cat(all_final_scores).numpy()\n        losses = torch.cat(all_losses).numpy()\n        correctness = torch.cat(all_correct).numpy()\n        groups = torch.cat(all_groups).numpy()\n        labels = torch.cat(all_labels).numpy()\n        attrs = torch.cat(all_attrs).numpy()\n\n        def _corr(x, y):\n            if len(x) >= 2 and np.std(x) > 0 and np.std(y) > 0:\n                return (\n                    float(stats.spearmanr(x, y).statistic),\n                    float(stats.pearsonr(x, y).statistic),\n                )\n            return 0.0, 0.0\n\n        spearman_loss, pearson_loss = _corr(scores, losses)\n        spearman_final_loss, pearson_final_loss = _corr(final_scores, losses)\n\n        if len(scores) >= 2 and np.std(scores) > 0 and np.std(correctness) > 0:\n            spearman_correct = float(stats.spearmanr(scores, correctness).statistic)\n        else:\n            spearman_correct = 0.0\n\n        if len(final_scores) >= 2 and np.std(final_scores) > 0 and np.std(correctness) > 0:\n            spearman_final_correct = float(\n                stats.spearmanr(final_scores, correctness).statistic\n            )\n        else:\n            spearman_final_correct = 0.0\n\n        return {\n            "scores": scores,\n            "final_scores": final_scores,\n            "training_weights": final_scores,\n            "mean_loss": losses,\n            "mean_correctness": correctness,\n            "groups": groups,\n            "labels": labels,\n            "attrs": attrs,\n            "spearman_score_vs_loss": spearman_loss,\n            "pearson_score_vs_loss": pearson_loss,\n            "spearman_score_vs_correctness": spearman_correct,\n            "spearman_final_score_vs_loss": spearman_final_loss,\n            "pearson_final_score_vs_loss": pearson_final_loss,\n            "spearman_final_score_vs_correctness": spearman_final_correct,\n        }\n\n    @torch.no_grad()\n    def _compute_scores(self, dataset, model=None):\n        """\n        Save raw Rater scores and, when a classifier is supplied, the\n        effective EA-GATED Rater weights for that classifier.\n        """\n\n        loader = DataLoader(\n            dataset,\n            batch_size=self.config.batch_size,\n            shuffle=False,\n            num_workers=self.config.num_workers,\n            pin_memory=True,\n        )\n\n        self.rater.eval()\n        if model is not None:\n            model.eval()\n\n        scores, final_scores = [], []\n        labels, groups, attrs, correctness = [], [], [], []\n\n        for z, y, g, a in loader:\n            z = z.to(self.device, non_blocking=True)\n            y_device = y.to(self.device, non_blocking=True)\n            batch_scores = self.rater(z)\n\n            if model is None:\n                batch_final_scores = self._scores_to_weights(batch_scores)\n                batch_correctness = torch.full(\n                    (y_device.shape[0],),\n                    float("nan"),\n                    device=self.device,\n                )\n            else:\n                logits = model(z)\n                batch_final_scores, correct_mask, _ = (\n                    self._all_sample_training_weights(\n                        z=z,\n                        y=y_device,\n                        logits=logits,\n                        raw_scores_all=batch_scores,\n                    )\n                )\n                batch_correctness = correct_mask.float()\n\n            scores.append(batch_scores.cpu())\n            final_scores.append(batch_final_scores.cpu())\n            correctness.append(batch_correctness.cpu())\n            labels.append(y.cpu())\n            groups.append(g.cpu())\n            attrs.append(a.cpu())\n\n        return {\n            "scores": torch.cat(scores),\n            "final_scores": torch.cat(final_scores),\n            "training_weights": torch.cat(final_scores),\n            "correctness": torch.cat(correctness),\n            "labels": torch.cat(labels),\n            "groups": torch.cat(groups),\n            "attrs": torch.cat(attrs),\n            "weights_are_all_sample_rater": model is not None,\n        }\n\n    def _rating_summary(self, payload):\n        scores = payload["scores"]\n        final_scores = payload.get("final_scores", None)\n        labels = payload["labels"]\n        groups = payload["groups"]\n\n        lines = [\n            f"raw score mean={scores.mean():.6f}, "\n            f"std={scores.std():.6f}, "\n            f"min={scores.min():.6f}, "\n            f"max={scores.max():.6f}"\n        ]\n\n        if final_scores is not None:\n            lines.append(\n                f"saved final/effective weight mean="\n                f"{final_scores.mean():.6f}, "\n                f"std={final_scores.std():.6f}, "\n                f"min={final_scores.min():.6f}, "\n                f"max={final_scores.max():.6f}"\n            )\n\n        for c in torch.unique(labels):\n            mask = labels == c\n            msg = (\n                f"class {int(c)}: n={int(mask.sum())}, "\n                f"mean_raw_score={scores[mask].mean():.6f}"\n            )\n            if final_scores is not None:\n                msg += (\n                    f", mean_final_score="\n                    f"{final_scores[mask].mean():.6f}"\n                )\n            lines.append(msg)\n\n        for g in torch.unique(groups):\n            mask = groups == g\n            msg = (\n                f"group {int(g)}: n={int(mask.sum())}, "\n                f"mean_raw_score={scores[mask].mean():.6f}"\n            )\n            if final_scores is not None:\n                msg += (\n                    f", mean_final_score="\n                    f"{final_scores[mask].mean():.6f}"\n                )\n            lines.append(msg)\n\n        return lines\n\n    # ========================================================\n    # Checkpoints\n    # ========================================================\n\n    def _save_rater_checkpoint(\n        self,\n        path,\n        meta_step,\n        meta_loss,\n        outer_ce=None,\n        grad_mse=None,\n    ):\n        torch.save(\n            {\n                "rater_sd": self.rater.state_dict(),\n                "outer_optimizer_sd": (\n                    self.outer_optimizer.state_dict()\n                ),\n                "meta_step": meta_step,\n                "meta_loss": meta_loss,\n\n                # Backward-compatible key.\n                "outer_loss": meta_loss,\n\n                "outer_ce": outer_ce,\n                "grad_mse": 0.0,\n                "lambda_g": 0.0,\n                "meta_objective": (\n                    "outer_classification_cross_entropy_only"\n                ),\n                "rate_all_samples": True,\n                "inner_initialization": (\n                    "erm_head_exact_for_model0_"\n                    "plus_small_noise_for_models1plus"\n                ),\n                "num_inner_models": self.num_inner_models,\n                "inner_init_noise_std": (\n                    self.inner_init_noise_std\n                ),\n                "erm_model_path": self.erm_model_path,\n                "erm_checkpoint_fingerprint": (\n                    self.erm_checkpoint_fingerprint\n                ),\n                "erm_checkpoint_epoch": (\n                    self.erm_checkpoint_epoch\n                ),\n\n                "feature_dim": self.feature_dim,\n                "backbone": self.config.backbone,\n                "pretrained": True,\n                "rater_capacity": self.rater_capacity,\n                "rater_weighting": self.weighting,\n                "rater_temperature": self.temperature,\n\n                "embedding_cache_root": (\n                    self.embedding_cache_root\n                ),\n                "embedding_cache_namespace": (\n                    self._embedding_cache_namespace()\n                ),\n            },\n            path,\n        )\n\n    def _load_rater_checkpoint(self, path):\n        if not os.path.exists(path):\n            raise ValueError(f"Rater checkpoint does not exist: {path}")\n\n        checkpoint = torch.load(\n            path,\n            map_location="cpu",\n            weights_only=False,\n        )\n\n        if "rater_sd" in checkpoint:\n            self.rater.load_state_dict(checkpoint["rater_sd"])\n        else:\n            # Also permit loading a raw rater state_dict.\n            self.rater.load_state_dict(checkpoint)\n\n        log(f"[Rater] Loaded checkpoint from {path}")\n\n    def _save_population_snapshot(self, output_dir, meta_step):\n        """Save raw state_dict snapshots for post-hoc evaluation."""\n\n        rater_dir = os.path.join(output_dir, "rater_checkpoints")\n        model_dir = os.path.join(output_dir, "models")\n        os.makedirs(rater_dir, exist_ok=True)\n        os.makedirs(model_dir, exist_ok=True)\n\n        torch.save(\n            self.rater.state_dict(),\n            os.path.join(\n                rater_dir,\n                f"data_rater_{meta_step:06d}.pt",\n            ),\n        )\n\n        for i, model in enumerate(self.inner_models):\n            torch.save(\n                model.state_dict(),\n                os.path.join(\n                    model_dir,\n                    f"inner_model{i}_{meta_step:06d}.pt",\n                ),\n            )\n\n    def _save_final_classifier(self, path, model):\n        torch.save(\n            {\n                "model_sd": model.state_dict(),\n                "feature_dim": self.feature_dim,\n                "num_classes": self.n_classes,\n            },\n            path,\n        )\n\n    def _load_final_classifier(self, path):\n        checkpoint = torch.load(\n            path,\n            map_location="cpu",\n            weights_only=False,\n        )\n        model = self._new_final_classifier()\n        if "model_sd" in checkpoint:\n            model.load_state_dict(checkpoint["model_sd"])\n        else:\n            model.load_state_dict(checkpoint)\n        model.to(self.device)\n        model.eval()\n        return model\n\n    # ========================================================\n    # Split handling\n    # ========================================================\n\n    def _has_ea_calibration_split(self):\n        """\n        Return True when the validation set has been split into\n        val_subset1 / val_subset2.\n        """\n        return (\n            float(\n                getattr(\n                    self.config,\n                    "split_val",\n                    1.0,\n                )\n            ) < 1.0\n            and "val_subset1" in self.dataloaders\n            and "val_subset2" in self.dataloaders\n        )\n\n    def _resolve_meta_splits(self, split):\n        """\n        Rater bilevel data protocol.\n\n        EA-style calibration mode (--split_val < 1):\n            inner/meta-train = train_no_aug\n            outer/meta-val   = val_subset1\n\n        This reserves val_subset2 for final-classifier model selection.\n\n        Legacy mode (--split_val == 1):\n            inner/meta-train = train_no_aug\n            outer/meta-val   = val\n        """\n\n        if split in {"train", "train_no_aug"}:\n            inner_split = "train_no_aug"\n\n            if (\n                self.use_calibration_final\n                and self._has_ea_calibration_split()\n            ):\n                return inner_split, "val_subset1"\n\n            return inner_split, "val"\n\n        if split == "train_subset1":\n            if "train_no_aug_subset1" in self.dataloaders:\n                inner_split = "train_no_aug_subset1"\n            else:\n                inner_split = split\n\n            if (\n                self.use_calibration_final\n                and self._has_ea_calibration_split()\n            ):\n                return inner_split, "val_subset1"\n\n            return inner_split, "val"\n\n        if split == "val_subset1":\n            if "val_subset2" not in self.dataloaders:\n                raise ValueError(\n                    "val_subset1 requires --split_val < 1 so that "\n                    "val_subset2 exists."\n                )\n            return "val_subset1", "val_subset2"\n\n        return split, "val"\n\n    def _resolve_final_classifier_splits(self):\n        """\n        Final classifier protocol.\n\n        EA-style:\n            calibration / classifier training = val_subset1\n            checkpoint selection              = val_subset2\n            final reporting                   = test\n        """\n        if self.use_calibration_final:\n            if not self._has_ea_calibration_split():\n                raise ValueError(\n                    "EA-style final-classifier calibration is enabled, "\n                    "but val_subset1/val_subset2 do not exist. "\n                    "Run with --split_val 0.4 (the EA repository example) "\n                    "or another value < 1."\n                )\n\n            return "val_subset1", "val_subset2"\n\n        return "train_no_aug", "val"\n\n    # ========================================================\n    # Final-classifier epoch diagnostics\n    # ========================================================\n\n    def _save_final_epoch_weight_histograms(\n        self,\n        output_dir,\n        epoch,\n        weights,\n        groups,\n        correctness,\n        split_name="train",\n    ):\n        """\n        Save ACTUAL EA-GATED Rater weights used during one\n        final-classifier epoch.\n\n        Correctness is diagnostic only; it does not gate the weights.\n        """\n        weights = np.asarray(\n            weights,\n            dtype=np.float64,\n        )\n\n        groups = np.asarray(\n            groups,\n            dtype=np.int64,\n        )\n\n        correctness = np.asarray(\n            correctness,\n            dtype=np.float64,\n        )\n\n        if len(weights) == 0:\n            return\n\n        data_dir = os.path.join(\n            output_dir,\n            "plots",\n            "final_classifier_epoch_weight_data",\n        )\n\n        os.makedirs(\n            data_dir,\n            exist_ok=True,\n        )\n\n        csv_path = os.path.join(\n            data_dir,\n            f"epoch_{epoch:03d}_{split_name}_weights.csv",\n        )\n\n        with open(\n            csv_path,\n            "w",\n            newline="",\n        ) as f:\n            writer = csv.writer(f)\n\n            writer.writerow(\n                [\n                    "index",\n                    "all_sample_rater_weight",\n                    "group",\n                    "correct_before_update",\n                ]\n            )\n\n            for i in range(\n                len(weights)\n            ):\n                writer.writerow(\n                    [\n                        i,\n                        float(weights[i]),\n                        int(groups[i]),\n                        int(correctness[i] >= 0.5),\n                    ]\n                )\n\n        plot_dir = os.path.join(\n            output_dir,\n            "plots",\n            "final_classifier_epoch_all_sample_weight_histogram",\n        )\n\n        os.makedirs(\n            plot_dir,\n            exist_ok=True,\n        )\n\n        finite = weights[\n            np.isfinite(weights)\n        ]\n\n        if len(finite) == 0:\n            return\n\n        wmin = float(\n            finite.min()\n        )\n\n        wmax = float(\n            finite.max()\n        )\n\n        if np.isclose(\n            wmin,\n            wmax,\n        ):\n            eps = max(\n                abs(wmin) * 0.05,\n                1e-6,\n            )\n\n            bins = np.linspace(\n                wmin - eps,\n                wmax + eps,\n                30,\n            )\n        else:\n            bins = np.linspace(\n                wmin,\n                wmax,\n                41,\n            )\n\n        fig, ax = plt.subplots(\n            figsize=(10, 7)\n        )\n\n        for gid in sorted(\n            np.unique(groups)\n        ):\n            mask = groups == gid\n\n            if not np.any(mask):\n                continue\n\n            ax.hist(\n                weights[mask],\n                bins=bins,\n                density=True,\n                alpha=0.45,\n                label=(\n                    f"{self._group_display_name(gid)} "\n                    f"(n={int(mask.sum())})"\n                ),\n            )\n\n        ax.set_xlabel(\n            f"EA-GATED trained-Rater {self.weighting} weight"\n        )\n\n        ax.set_ylabel(\n            "Density"\n        )\n\n        ax.set_title(\n            f"Final-classifier EA-GATED training weights by group | "\n            f"epoch {epoch}\\\\n"\n            f"correct before update="\n            f"{100.0 * correctness.mean():.2f}%"\n        )\n\n        ax.grid(\n            True,\n            linestyle=":",\n            alpha=0.30,\n        )\n\n        ax.legend(\n            loc="best",\n            fontsize=9,\n        )\n\n        fig.tight_layout()\n\n        plot_path = os.path.join(\n            plot_dir,\n            f"epoch_{epoch:03d}_{split_name}_all_sample_weight_hist.png",\n        )\n\n        fig.savefig(\n            plot_path,\n            dpi=160,\n            bbox_inches="tight",\n        )\n\n        plt.close(fig)\n\n        summary_parts = []\n\n        for gid in sorted(\n            np.unique(groups)\n        ):\n            mask = groups == gid\n\n            summary_parts.append(\n                f"g{int(gid)}: "\n                f"mean_w={weights[mask].mean():.3f}, "\n                f"std_w={weights[mask].std():.3f}, "\n                f"acc_before_update="\n                f"{100.0 * correctness[mask].mean():.1f}%"\n            )\n\n        log(\n            f"[Final classifier epoch {epoch:03d} EA-GATED weights] "\n            + "; ".join(summary_parts)\n        )\n\n        log(\n            f"[Final classifier plot] saved EA-GATED weight histogram: "\n            f"{plot_path}"\n        )\n\n    # ========================================================\n    # Final weighted classifier\n    # ========================================================\n\n    def _selection_value(self, metrics):\n        if self.final_selection == "accuracy":\n            return metrics["accuracy"]\n        if self.final_selection == "wga":\n            return metrics["worst_group_accuracy"]\n        # Lower loss is better, so negate it to retain max-selection logic.\n        return -metrics["loss"]\n\n    def _train_final_classifier(\n        self,\n        calibration_dataset,\n        selection_dataset,\n        test_dataset,\n        output_dir,\n        calibration_split="train_no_aug",\n        selection_split="val",\n        test_split="test",\n    ):\n        """\n        Train a fresh linear classifier with the trained Rater frozen.\n\n        EVERY calibration sample receives a Rater-derived weight:\n\n            raw_i = frozen_Rater(z_i)\n            w_i   = selected transform(raw_i)\n\n            L = sum_i w_i * CE_i\n\n        There is NO correct/misclassified gate and NO division by\n        sum(weights).\n        """\n        log(\n            "[Rater] Training fresh final classifier with "\n            "FROZEN trained Rater and EA-GATED weighting..."\n        )\n\n        log(\n            f"[Final classifier] calibration/train = "\n            f"{calibration_split} "\n            f"({len(calibration_dataset)} samples)"\n        )\n\n        log(\n            f"[Final classifier] selection = "\n            f"{selection_split} "\n            f"({len(selection_dataset)} samples)"\n        )\n\n        log(\n            f"[Final classifier] test diagnostic = "\n            f"{test_split} "\n            f"({len(test_dataset)} samples)"\n        )\n\n        log(\n            f"[Final classifier] selection metric = "\n            f"{self.final_selection}"\n        )\n\n        log(\n            "[Final classifier] EVERY sample uses a frozen-Rater weight."\n        )\n\n        self.rater.eval()\n\n        for p in self.rater.parameters():\n            p.requires_grad_(False)\n\n        model = self._new_final_classifier()\n\n        optimizer = torch.optim.SGD(\n            model.parameters(),\n            lr=self.final_lr,\n            momentum=self.final_momentum,\n            weight_decay=self.final_weight_decay,\n        )\n\n        train_loader = DataLoader(\n            calibration_dataset,\n            batch_size=self.config.batch_size,\n            shuffle=True,\n            num_workers=self.config.num_workers,\n            pin_memory=True,\n        )\n\n        best_value = -float("inf")\n        best_state = None\n        best_epoch = None\n        best_metrics = None\n\n        history = []\n\n        for epoch in range(\n            1,\n            self.final_epochs + 1,\n        ):\n            model.train()\n\n            running_weighted_loss = 0.0\n            total_correct = 0\n            total_examples = 0\n            num_batches = 0\n\n            epoch_weights = []\n            epoch_groups = []\n            epoch_correctness = []\n\n            for z, y, g, _ in train_loader:\n                z = z.to(\n                    self.device,\n                    non_blocking=True,\n                )\n\n                y = y.to(\n                    self.device,\n                    non_blocking=True,\n                )\n\n                g = torch.as_tensor(\n                    g\n                ).to(\n                    self.device,\n                    non_blocking=True,\n                )\n\n                logits = model(z)\n\n                with torch.no_grad():\n                    raw_scores = self.rater(z)\n\n                    weights = self._scores_to_weights(\n                        raw_scores\n                    )\n\n                    correct_mask = (\n                        logits.argmax(dim=1).eq(y)\n                    )\n\n                epoch_weights.append(\n                    weights.detach().cpu()\n                )\n\n                epoch_groups.append(\n                    g.detach().cpu()\n                )\n\n                epoch_correctness.append(\n                    correct_mask.float()\n                    .detach()\n                    .cpu()\n                )\n\n                per_sample_loss = F.cross_entropy(\n                    logits,\n                    y,\n                    reduction="none",\n                )\n\n                weighted_loss = (\n                    per_sample_loss\n                    * weights\n                ).sum()\n\n                optimizer.zero_grad()\n\n                weighted_loss.backward()\n\n                optimizer.step()\n\n                running_weighted_loss += float(\n                    weighted_loss.item()\n                )\n\n                total_correct += int(\n                    correct_mask.sum().item()\n                )\n\n                total_examples += int(\n                    y.numel()\n                )\n\n                num_batches += 1\n\n            train_acc = (\n                total_correct\n                / max(total_examples, 1)\n            )\n\n            train_weighted_loss = (\n                running_weighted_loss\n                / max(num_batches, 1)\n            )\n\n            epoch_weights_np = torch.cat(\n                epoch_weights\n            ).numpy()\n\n            epoch_groups_np = torch.cat(\n                epoch_groups\n            ).numpy()\n\n            epoch_correctness_np = torch.cat(\n                epoch_correctness\n            ).numpy()\n\n            should_plot_final = (\n                epoch == 1\n                or epoch % self.final_plot_freq == 0\n                or epoch == self.final_epochs\n            )\n\n            if should_plot_final:\n                self._save_final_epoch_weight_histograms(\n                    output_dir=output_dir,\n                    epoch=epoch,\n                    weights=epoch_weights_np,\n                    groups=epoch_groups_np,\n                    correctness=epoch_correctness_np,\n                    split_name=calibration_split,\n                )\n\n            selection_metrics = (\n                self._evaluate_classifier(\n                    model,\n                    selection_dataset,\n                )\n            )\n\n            selection_value = (\n                self._selection_value(\n                    selection_metrics\n                )\n            )\n\n            test_metrics = (\n                self._evaluate_classifier(\n                    model,\n                    test_dataset,\n                )\n            )\n\n            history.append(\n                [\n                    epoch,\n                    train_weighted_loss,\n                    train_acc,\n                    selection_metrics["loss"],\n                    selection_metrics["accuracy"],\n                    selection_metrics["worst_group_accuracy"],\n                    test_metrics["loss"],\n                    test_metrics["accuracy"],\n                    test_metrics["worst_group_accuracy"],\n                    float(epoch_weights_np.mean()),\n                    float(epoch_weights_np.std()),\n                ]\n            )\n\n            log(\n                f"[Final classifier epoch {epoch:03d}] "\n                f"weighted_train_loss={train_weighted_loss:.6f}, "\n                f"train_acc={100.0 * train_acc:.2f}%, "\n                f"mean_all_sample_weight={epoch_weights_np.mean():.4f}, "\n                f"selection_acc="\n                f"{100.0 * selection_metrics[\'accuracy\']:.2f}%, "\n                f"selection_WGA="\n                f"{100.0 * selection_metrics[\'worst_group_accuracy\']:.2f}%, "\n                f"test_acc={100.0 * test_metrics[\'accuracy\']:.2f}%, "\n                f"test_WGA="\n                f"{100.0 * test_metrics[\'worst_group_accuracy\']:.2f}%"\n            )\n\n            if selection_value > best_value:\n                best_value = selection_value\n                best_epoch = epoch\n\n                best_metrics = {\n                    "loss": float(\n                        selection_metrics["loss"]\n                    ),\n                    "accuracy": float(\n                        selection_metrics["accuracy"]\n                    ),\n                    "worst_group_accuracy": float(\n                        selection_metrics[\n                            "worst_group_accuracy"\n                        ]\n                    ),\n                    "group_accuracy": {\n                        int(k): float(v)\n                        for k, v\n                        in selection_metrics[\n                            "group_accuracy"\n                        ].items()\n                    },\n                }\n\n                best_state = {\n                    k: v.detach().cpu().clone()\n                    for k, v\n                    in model.state_dict().items()\n                }\n\n                torch.save(\n                    {\n                        "model_sd": best_state,\n                        "epoch": best_epoch,\n                        "selection_metric": self.final_selection,\n                        "selection_value": float(best_value),\n                        "selection_metrics": best_metrics,\n                        "all_sample_rater_weighting": True,\n                    },\n                    os.path.join(\n                        output_dir,\n                        "best_final_weighted_classifier.pt",\n                    ),\n                )\n\n        if best_state is None:\n            raise RuntimeError(\n                "Final classifier training produced no model."\n            )\n\n        model.load_state_dict(\n            best_state\n        )\n\n        model.to(\n            self.device\n        )\n\n        model.eval()\n\n        history_path = os.path.join(\n            output_dir,\n            "final_classifier_history.csv",\n        )\n\n        with open(\n            history_path,\n            "w",\n            newline="",\n        ) as f:\n            writer = csv.writer(f)\n\n            writer.writerow(\n                [\n                    "epoch",\n                    "weighted_train_loss",\n                    "train_accuracy",\n                    "selection_loss",\n                    "selection_accuracy",\n                    "selection_wga",\n                    "test_loss",\n                    "test_accuracy",\n                    "test_wga",\n                    "mean_all_sample_weight",\n                    "std_all_sample_weight",\n                ]\n            )\n\n            writer.writerows(\n                history\n            )\n\n        log(\n            f"[Final classifier] Restored selected epoch "\n            f"{best_epoch:03d}."\n        )\n\n        self.rater.eval()\n\n        return model\n\n\n    # ========================================================\n    # Meta-only experiment helpers\n    # ========================================================\n\n    def _meta_log(self, message):\n        log(message)\n\n        path = getattr(\n            self,\n            "meta_log_path",\n            None,\n        )\n\n        if path:\n            with open(\n                path,\n                "a",\n                encoding="utf-8",\n            ) as fout:\n                fout.write(\n                    str(message)\n                    + "\\n"\n                )\n\n    def _group_name_meta(self, gid):\n        names = {\n            0: "g0: landbird / land",\n            1: "g1: landbird / water",\n            2: "g2: waterbird / land",\n            3: "g3: waterbird / water",\n        }\n\n        return names.get(\n            int(gid),\n            f"g{int(gid)}",\n        )\n\n    def _save_meta_rate_box_ecdf(\n        self,\n        relationship,\n        output_dir,\n        tag,\n        split_name,\n    ):\n        plot_dir = os.path.join(\n            output_dir,\n            "plots",\n            "meta_rate_distributions",\n        )\n\n        os.makedirs(\n            plot_dir,\n            exist_ok=True,\n        )\n\n        groups = np.asarray(\n            relationship["groups"]\n        )\n\n        for key, ylabel, suffix in [\n            (\n                "scores",\n                "Raw Rater score",\n                "raw",\n            ),\n            (\n                "final_scores",\n                f"Effective {self.weighting} weight",\n                "weight",\n            ),\n        ]:\n            values = np.asarray(\n                relationship[key]\n            )\n\n            gids = sorted(\n                np.unique(groups)\n            )\n\n            data = [\n                values[groups == gid]\n                for gid in gids\n            ]\n\n            labels = [\n                self._group_name_meta(gid)\n                for gid in gids\n            ]\n\n            fig, ax = plt.subplots(\n                figsize=(11, 7)\n            )\n\n            try:\n                ax.boxplot(\n                    data,\n                    tick_labels=labels,\n                    showfliers=False,\n                )\n            except TypeError:\n                ax.boxplot(\n                    data,\n                    labels=labels,\n                    showfliers=False,\n                )\n\n            ax.set_ylabel(ylabel)\n            ax.set_title(\n                f"{ylabel} by group | {split_name} | {tag}"\n            )\n            ax.tick_params(\n                axis="x",\n                labelrotation=20,\n            )\n            ax.grid(\n                True,\n                axis="y",\n                linestyle=":",\n                alpha=0.30,\n            )\n            fig.tight_layout()\n            fig.savefig(\n                os.path.join(\n                    plot_dir,\n                    f"{tag}_{split_name}_{suffix}_box.png",\n                ),\n                dpi=160,\n                bbox_inches="tight",\n            )\n            plt.close(fig)\n\n            fig, ax = plt.subplots(\n                figsize=(10, 7)\n            )\n\n            for gid in gids:\n                vals = np.sort(\n                    values[groups == gid]\n                )\n\n                if len(vals) == 0:\n                    continue\n\n                yy = np.arange(\n                    1,\n                    len(vals) + 1,\n                    dtype=np.float64,\n                ) / len(vals)\n\n                ax.step(\n                    vals,\n                    yy,\n                    where="post",\n                    linewidth=2,\n                    label=self._group_name_meta(gid),\n                )\n\n            ax.set_xlabel(ylabel)\n            ax.set_ylabel("Empirical CDF")\n            ax.set_title(\n                f"{ylabel} ECDF by group | {split_name} | {tag}"\n            )\n            ax.grid(\n                True,\n                linestyle=":",\n                alpha=0.30,\n            )\n            ax.legend(fontsize=9)\n            fig.tight_layout()\n            fig.savefig(\n                os.path.join(\n                    plot_dir,\n                    f"{tag}_{split_name}_{suffix}_ecdf.png",\n                ),\n                dpi=160,\n                bbox_inches="tight",\n            )\n            plt.close(fig)\n\n    def _save_meta_training_curves(\n        self,\n        history,\n        group_trajectory,\n        output_dir,\n    ):\n        plot_dir = os.path.join(\n            output_dir,\n            "plots",\n            "meta_training_curves",\n        )\n\n        os.makedirs(\n            plot_dir,\n            exist_ok=True,\n        )\n\n        if history:\n            steps = np.asarray(\n                [row["step"] for row in history]\n            )\n            outer_ce = np.asarray(\n                [row["outer_ce"] for row in history]\n            )\n\n            fig, ax = plt.subplots(\n                figsize=(10, 6)\n            )\n            ax.plot(\n                steps,\n                outer_ce,\n                linewidth=1.8,\n            )\n            ax.set_xlabel("Meta step")\n            ax.set_ylabel("Outer classification CE")\n            ax.set_title("Rater meta objective")\n            ax.grid(\n                True,\n                linestyle=":",\n                alpha=0.30,\n            )\n            fig.tight_layout()\n            fig.savefig(\n                os.path.join(\n                    plot_dir,\n                    "outer_classification_ce.png",\n                ),\n                dpi=160,\n                bbox_inches="tight",\n            )\n            plt.close(fig)\n\n        for field, ylabel, filename in [\n            (\n                "raw_mean",\n                "Mean raw Rater score",\n                "group_mean_raw_score.png",\n            ),\n            (\n                "weight_mean",\n                f"Mean {self.weighting} weight",\n                "group_mean_weight.png",\n            ),\n        ]:\n            if not group_trajectory:\n                continue\n\n            fig, ax = plt.subplots(\n                figsize=(11, 7)\n            )\n\n            gids = sorted(\n                {\n                    int(row["group"])\n                    for row in group_trajectory\n                }\n            )\n\n            for gid in gids:\n                rows = [\n                    row\n                    for row in group_trajectory\n                    if int(row["group"]) == gid\n                ]\n                rows = sorted(\n                    rows,\n                    key=lambda row: int(row["step"]),\n                )\n\n                ax.plot(\n                    [row["step"] for row in rows],\n                    [row[field] for row in rows],\n                    marker="o",\n                    markersize=3,\n                    linewidth=1.5,\n                    label=self._group_name_meta(gid),\n                )\n\n            ax.set_xlabel("Meta step")\n            ax.set_ylabel(ylabel)\n            ax.set_title(\n                f"{ylabel} by group during meta-learning"\n            )\n            ax.grid(\n                True,\n                linestyle=":",\n                alpha=0.30,\n            )\n            ax.legend(fontsize=9)\n            fig.tight_layout()\n            fig.savefig(\n                os.path.join(\n                    plot_dir,\n                    filename,\n                ),\n                dpi=160,\n                bbox_inches="tight",\n            )\n            plt.close(fig)\n\n    def _write_meta_summary(\n        self,\n        output_dir,\n        best_step,\n        best_outer_ce,\n        final_relationships,\n        final_metrics,\n    ):\n        lines = [\n            "RATER META-LEARNING SUMMARY",\n            "=" * 88,\n            "",\n            "Representation: frozen Waterbirds-ERM ResNet-50 embeddings",\n            f"ERM checkpoint: {self.erm_model_path}",\n            f"ERM fingerprint: {self.erm_checkpoint_fingerprint}",\n            f"Feature dimension: {self.feature_dim}",\n            "",\n            "Inner models: 2 independent random Linear(2048, 2) classifiers",\n            "Initialized from ERM classifier: NO",\n            "Reinitialized/refreshed during meta-training: NO",\n            (\n                "Before every Rater update: BOTH inner models each see "\n                "the ENTIRE train_no_aug set exactly once"\n            ),\n            "",\n            f"Meta steps: {self.meta_steps}",\n            f"Full inner steps per meta update: {self.inner_steps}",\n            f"Weighting transform for MISCLASSIFIED samples: {self.weighting}",\n            f"Temperature: {self.temperature}",\n            "EA-style gate: correct -> 1; wrong -> Rater rating",\n            f"Inner LR: {self.inner_lr}",\n            f"Outer LR: {self.outer_lr}",\n            f"Gradient clip: {self.grad_clip}",\n            "Inner objective: SUM_i weight_i * CE_i",\n            "Divide by sum(weights): NO",\n            "Rater meta objective: OUTER CLASSIFICATION CE ONLY",\n            f"Best meta step: {best_step}",\n            f"Best outer CE: {best_outer_ce:.8f}",\n            "",\n            "FINAL INNER-POPULATION METRICS",\n            "-" * 88,\n        ]\n\n        for split_name, metrics in final_metrics.items():\n            lines.append(\n                f"{split_name}: "\n                f"mean_loss={metrics[\'mean_loss\']:.6f}, "\n                f"mean_acc={metrics[\'mean_accuracy\']:.6f}, "\n                f"mean_WGA={metrics[\'mean_wga\']:.6f}"\n            )\n\n        lines.extend(\n            [\n                "",\n                "FINAL RATER RATE/WEIGHT SUMMARIES",\n                "-" * 88,\n            ]\n        )\n\n        for split_name, relationship in final_relationships.items():\n            scores = np.asarray(\n                relationship["scores"]\n            )\n            weights = np.asarray(\n                relationship["final_scores"]\n            )\n            losses = np.asarray(\n                relationship["mean_loss"]\n            )\n            groups = np.asarray(\n                relationship["groups"]\n            )\n\n            lines.append(\n                f"{split_name}: n={len(scores)}, "\n                f"raw_mean={scores.mean():.6f}, "\n                f"raw_std={scores.std():.6f}, "\n                f"weight_mean={weights.mean():.6f}, "\n                f"weight_std={weights.std():.6f}"\n            )\n\n            for gid in sorted(np.unique(groups)):\n                mask = groups == gid\n                lines.append(\n                    f"  {self._group_name_meta(gid)}: "\n                    f"n={int(mask.sum())}, "\n                    f"raw_mean={scores[mask].mean():.6f}, "\n                    f"weight_mean={weights[mask].mean():.6f}, "\n                    f"loss_mean={losses[mask].mean():.6f}"\n                )\n\n        lines.extend(\n            [\n                "",\n                "Saved outputs:",\n                "  plots/",\n                "  log.txt",\n                "  summary.txt",\n                "  final_rater.pt",\n            ]\n        )\n\n        path = os.path.join(\n            output_dir,\n            "summary.txt",\n        )\n\n        with open(\n            path,\n            "w",\n            encoding="utf-8",\n        ) as fout:\n            fout.write(\n                "\\n".join(lines)\n                + "\\n"\n            )\n\n        return path\n\n    # ========================================================\n    # Main training -- FROZEN RATER / FINAL CLASSIFIER ONLY\n    # ========================================================\n\n    def train(\n        self,\n        output_dir,\n        split="train",\n    ):\n        """\n        Train ONLY the Rater.\n\n        No final classifier is trained here.\n\n        Crucial full-epoch rule:\n            self.inner_steps = len(train_loader)\n\n        Since _meta_step unrolls inner model 0 for self.inner_steps batches\n        and then inner model 1 for self.inner_steps batches BEFORE the single\n        outer/Rater update, each inner classifier sees every training sample\n        exactly once before every Rater update.\n        """\n        os.makedirs(\n            output_dir,\n            exist_ok=True,\n        )\n\n        self.meta_log_path = os.path.join(\n            output_dir,\n            "log.txt",\n        )\n\n        with open(\n            self.meta_log_path,\n            "w",\n            encoding="utf-8",\n        ) as fout:\n            fout.write(\n                "Rater meta-learning log\\n"\n            )\n\n        if self.precompute_embeddings_only:\n            self._precompute_shared_embeddings(\n                output_dir\n            )\n            return\n\n        cache_dir = self._resolve_embedding_cache_dir(\n            output_dir\n        )\n\n        inner_split, outer_split = self._resolve_meta_splits(\n            split\n        )\n\n        self._meta_log(\n            f"[Rater] inner/meta-train={inner_split}"\n        )\n        self._meta_log(\n            f"[Rater] outer/meta-loss={outer_split}"\n        )\n\n        train_dataset = self._extract_embedding_dataset(\n            inner_split,\n            cache_dir,\n        )\n        val_dataset = self._extract_embedding_dataset(\n            outer_split,\n            cache_dir,\n        )\n        test_dataset = self._extract_embedding_dataset(\n            "test",\n            cache_dir,\n        )\n\n        untouched_val_dataset = None\n        if (\n            float(getattr(self.config, "split_val", 1.0)) < 1.0\n            and "val_subset2" in self.dataloaders\n        ):\n            untouched_val_dataset = self._extract_embedding_dataset(\n                "val_subset2",\n                cache_dir,\n            )\n\n        # All needed embeddings are now in CPU tensors. Release the frozen\n        # ResNet-50 so the 38-step differentiable inner unroll has more GPU RAM.\n        if self.feature_model is not None:\n            self.feature_model.cpu()\n            del self.feature_model\n            self.feature_model = None\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n\n        train_loader = DataLoader(\n            train_dataset,\n            batch_size=self.config.batch_size,\n            shuffle=True,\n            num_workers=self.config.num_workers,\n            pin_memory=True,\n        )\n        val_loader = DataLoader(\n            val_dataset,\n            batch_size=self.config.batch_size,\n            shuffle=True,\n            num_workers=self.config.num_workers,\n            pin_memory=True,\n        )\n\n        # ----------------------------------------------------\n        # FULL-EPOCH GUARANTEE\n        # ----------------------------------------------------\n        # Waterbirds: 4795 / 128 -> 38 batches.\n        # This is dynamic, so it stays correct if the split changes.\n        # ----------------------------------------------------\n        self.inner_steps = len(train_loader)\n\n        self._meta_log(\n            f"[Rater] train_n={len(train_dataset)}, "\n            f"batch_size={self.config.batch_size}, "\n            f"full_inner_steps={self.inner_steps}"\n        )\n        self._meta_log(\n            "[Rater] Before EVERY Rater update, BOTH inner models "\n            "see the complete training set exactly once."\n        )\n        self._meta_log(\n            "[Rater] Inner classifiers are random from scratch and "\n            "are NEVER refreshed during the 500 meta steps."\n        )\n\n        train_iterator = iter(train_loader)\n        val_iterator = iter(val_loader)\n\n        self._initialize_inner_population()\n\n        best_outer_ce = float("inf")\n        best_step = None\n        best_rater_state = None\n\n        history = []\n        group_trajectory = []\n\n        self._meta_log(\n            "[Rater] EA-style gate: correct -> 1; "\n            "misclassified -> Rater rating."\n        )\n        self._meta_log(\n            f"[Rater] meta_steps={self.meta_steps}, "\n            f"inner_models={self.num_inner_models}, "\n            f"misclassified_weighting={self.weighting}, "\n            f"inner_lr={self.inner_lr}, "\n            f"outer_lr={self.outer_lr}, "\n            f"temperature={self.temperature}, "\n            f"grad_clip={self.grad_clip}"\n        )\n\n        for meta_step in tqdm(\n            range(1, self.meta_steps + 1),\n            desc="Rater full-epoch meta-training",\n        ):\n            (\n                total_meta_loss,\n                outer_ce,\n                _,\n                train_iterator,\n                val_iterator,\n            ) = self._meta_step(\n                train_iterator,\n                val_iterator,\n                train_loader,\n                val_loader,\n            )\n\n            history.append(\n                {\n                    "step": int(meta_step),\n                    "outer_ce": float(outer_ce),\n                }\n            )\n\n            if outer_ce < best_outer_ce:\n                best_outer_ce = float(outer_ce)\n                best_step = int(meta_step)\n                best_rater_state = {\n                    key: value.detach().cpu().clone()\n                    for key, value in self.rater.state_dict().items()\n                }\n\n            should_eval = (\n                meta_step == 1\n                or meta_step % self.eval_freq == 0\n                or meta_step == self.meta_steps\n            )\n            should_plot = (\n                meta_step == 1\n                or meta_step % self.plot_freq == 0\n                or meta_step == self.meta_steps\n            )\n\n            if should_eval or should_plot:\n                population_metrics = self._evaluate_inner_population(\n                    val_dataset,\n                    verbose=should_eval,\n                )\n                relationship = self._score_loss_relationship(\n                    val_dataset,\n                    models=self.inner_models,\n                )\n\n                scores = np.asarray(\n                    relationship["scores"]\n                )\n                weights = np.asarray(\n                    relationship["final_scores"]\n                )\n                groups = np.asarray(\n                    relationship["groups"]\n                )\n\n                for gid in sorted(np.unique(groups)):\n                    mask = groups == gid\n                    group_trajectory.append(\n                        {\n                            "step": int(meta_step),\n                            "group": int(gid),\n                            "raw_mean": float(scores[mask].mean()),\n                            "weight_mean": float(weights[mask].mean()),\n                        }\n                    )\n\n                self._meta_log(\n                    f"[step {meta_step:04d}] "\n                    f"outer_ce={outer_ce:.6f}, "\n                    f"mean_inner_acc="\n                    f"{100.0 * population_metrics[\'mean_accuracy\']:.2f}%, "\n                    f"mean_inner_WGA="\n                    f"{100.0 * population_metrics[\'mean_wga\']:.2f}%, "\n                    f"Spearman(score,loss)="\n                    f"{relationship[\'spearman_score_vs_loss\']:.4f}, "\n                    f"Spearman(weight,loss)="\n                    f"{relationship[\'spearman_final_score_vs_loss\']:.4f}"\n                )\n\n                if should_plot:\n                    tag = f"step_{meta_step:04d}"\n                    self._save_all_score_diagnostics(\n                        relationship=relationship,\n                        output_dir=output_dir,\n                        meta_step=meta_step,\n                        split_name=outer_split,\n                        tag_prefix="meta_step",\n                    )\n                    self._save_meta_rate_box_ecdf(\n                        relationship=relationship,\n                        output_dir=output_dir,\n                        tag=tag,\n                        split_name=outer_split,\n                    )\n\n        if best_rater_state is None:\n            raise RuntimeError(\n                "No valid Rater state was produced."\n            )\n\n        # Restore best Rater in memory.\n        self.rater.load_state_dict(\n            best_rater_state\n        )\n        self.rater.to(self.device)\n        self.rater.eval()\n\n        self._meta_log(\n            f"[Rater] restored best step={best_step}, "\n            f"best_outer_ce={best_outer_ce:.8f}"\n        )\n\n        # Save ONE compact reusable model.\n        final_rater_path = os.path.join(\n            output_dir,\n            "final_rater.pt",\n        )\n\n        torch.save(\n            {\n                "rater_sd": self.rater.state_dict(),\n                "meta_steps": int(self.meta_steps),\n                "best_meta_step": int(best_step),\n                "best_outer_ce": float(best_outer_ce),\n                "feature_dim": int(self.feature_dim),\n                "num_classes": int(self.n_classes),\n                "backbone": str(self.config.backbone),\n                "rater_capacity": str(self.rater_capacity),\n                "weighting": str(self.weighting),\n                "temperature": float(self.temperature),\n                "ea_style_gate": "correct=1; wrong=RaterRating",\n                "rater_applied_only_to_misclassified": True,\n                "inner_models": 2,\n                "inner_initialization": "independent_random_linear",\n                "full_train_epoch_before_each_meta_update": True,\n                "full_inner_steps": int(self.inner_steps),\n                "inner_lr": float(self.inner_lr),\n                "outer_lr": float(self.outer_lr),\n                "erm_model_path": str(self.erm_model_path),\n                "erm_fingerprint": str(self.erm_checkpoint_fingerprint),\n                "erm_epoch": self.erm_checkpoint_epoch,\n            },\n            final_rater_path,\n        )\n\n        self._save_meta_training_curves(\n            history,\n            group_trajectory,\n            output_dir,\n        )\n\n        # ----------------------------------------------------\n        # FINAL BEST-RATER diagnostics.\n        # ----------------------------------------------------\n        datasets = {\n            inner_split: train_dataset,\n            outer_split: val_dataset,\n            "test": test_dataset,\n        }\n\n        if untouched_val_dataset is not None:\n            datasets["val_subset2"] = untouched_val_dataset\n\n        final_relationships = {}\n        final_metrics = {}\n\n        for split_name, dataset in datasets.items():\n            relationship = self._score_loss_relationship(\n                dataset,\n                models=self.inner_models,\n            )\n            final_relationships[split_name] = relationship\n            final_metrics[split_name] = self._evaluate_inner_population(\n                dataset,\n                verbose=False,\n            )\n\n            self._save_all_score_diagnostics(\n                relationship=relationship,\n                output_dir=output_dir,\n                meta_step=self.meta_steps,\n                split_name=split_name,\n                tag_prefix="FINAL_BEST_RATER",\n            )\n            self._save_meta_rate_box_ecdf(\n                relationship=relationship,\n                output_dir=output_dir,\n                tag="FINAL_BEST_RATER",\n                split_name=split_name,\n            )\n\n            self._meta_log(\n                f"[FINAL {split_name}] "\n                f"raw_mean={np.asarray(relationship[\'scores\']).mean():.6f}, "\n                f"weight_mean="\n                f"{np.asarray(relationship[\'final_scores\']).mean():.6f}"\n            )\n\n        summary_path = self._write_meta_summary(\n            output_dir=output_dir,\n            best_step=best_step,\n            best_outer_ce=best_outer_ce,\n            final_relationships=final_relationships,\n            final_metrics=final_metrics,\n        )\n\n        self._meta_log(\n            f"[Rater] saved final_rater.pt={final_rater_path}"\n        )\n        self._meta_log(\n            f"[Rater] saved summary={summary_path}"\n        )\n\n    # ========================================================\n    # Test / final evaluation\n    # ========================================================\n\n    def test(\n        self,\n        output_dir,\n        split=("test",),\n        result_path="",\n    ):\n        del output_dir, split, result_path\n\n        if self.precompute_embeddings_only:\n            return\n\n        log(\n            "[Rater] Meta-learning-only run complete. "\n            "No final classifier is trained in this rater.py."\n        )\n\n'
rater_path.write_text(RATER_SOURCE, encoding="utf-8")
ast.parse(rater_path.read_text(encoding="utf-8"))

init_path = REPO_DIR / "algorithms" / "__init__.py"
init_text = init_path.read_text(encoding="utf-8")
if "from . import rater" not in init_text:
    init_path.write_text(init_text.rstrip() + "\nfrom . import rater\n", encoding="utf-8")

print("✓ meta-only rater.py installed")

✓ meta-only rater.py installed


In [6]:
# 6. Experiment parameters
META_STEPS = 500
BATCH_SIZE = 128
NUM_INNER_MODELS = 2

# Same EA-gated full-epoch sigmoid settings used before.
RATER_WEIGHTING = "sigmoid"
INNER_LR = 2e-4
TEMPERATURE = 1.0
OUTER_LR = 3e-4
GRAD_CLIP = 5.0

# Diagnostics: evaluate often, but avoid hundreds of PNGs.
EVAL_FREQ = 10
PLOT_FREQ = 25

os.environ.update({
    "RATER_ERM_MODEL": str(ERM_MODEL),
    "RATER_EMBEDDING_CACHE_ROOT": str(EMBED_ROOT),
    "RATER_META_STEPS": str(META_STEPS),
    "RATER_NUM_INNER_MODELS": str(NUM_INNER_MODELS),
    "RATER_WEIGHTING": RATER_WEIGHTING,
    "RATER_INNER_LR": str(INNER_LR),
    "RATER_OUTER_LR": str(OUTER_LR),
    "RATER_TEMPERATURE": str(TEMPERATURE),
    "RATER_GRAD_CLIP": str(GRAD_CLIP),
    "RATER_EVAL_FREQ": str(EVAL_FREQ),
    "RATER_PLOT_FREQ": str(PLOT_FREQ),
})

print({
    "meta_steps": META_STEPS,
    "batch_size": BATCH_SIZE,
    "inner_models": NUM_INNER_MODELS,
    "weighting": RATER_WEIGHTING,
    "inner_lr": INNER_LR,
    "outer_lr": OUTER_LR,
    "temperature": TEMPERATURE,
})

{'meta_steps': 500, 'batch_size': 128, 'inner_models': 2, 'weighting': 'sigmoid', 'inner_lr': 0.0002, 'outer_lr': 0.0003, 'temperature': 1.0}


In [7]:
# 7. Precompute/reuse persistent ERM embeddings
os.environ["RATER_PRECOMPUTE_EMBEDDINGS_ONLY"] = "1"
shutil.rmtree(LOCAL_PRE, ignore_errors=True)
LOCAL_PRE.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "main.py",
    "--dataset", "waterbirds",
    "--algorithm", "rater",
    "--backbone", "resnet50",
    "--pretrained", "True",
    "--erm_model", str(ERM_MODEL),
    "--save_folder", str(LOCAL_PRE),
    "--train_split", "train",
    "--split_val", "0.4",
    "--mode", "train",
    "--epoch", str(META_STEPS),
]

result = subprocess.run(cmd, cwd=str(REPO_DIR), env=os.environ.copy())
if result.returncode != 0:
    raise RuntimeError("Embedding preparation failed")

shutil.rmtree(LOCAL_PRE, ignore_errors=True)
print("✓ persistent ERM embeddings ready")

✓ persistent ERM embeddings ready


In [8]:
# 8. Verify cache
cache_files = sorted(EMBED_ROOT.rglob("*.pt"))
for p in cache_files:
    print(p.relative_to(EMBED_ROOT), f"{p.stat().st_size / 1024**2:.2f} MB")

assert any(p.name == "train_no_aug.pt" for p in cache_files)
assert any("val_subset1" in p.name for p in cache_files)
assert any("val_subset2" in p.name for p in cache_files)
assert any(p.name == "test.pt" for p in cache_files)

sample = next(p for p in cache_files if p.name == "train_no_aug.pt")
payload = torch.load(sample, map_location="cpu", weights_only=False)
assert payload["cache_metadata"]["representation_source"] == "waterbirds_erm_checkpoint"
print("✓ cache verified:", tuple(payload["embeddings"].shape))

waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/test.pt 45.40 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/train_no_aug.pt 37.57 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val.pt 9.40 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p4_seed_0.pt 3.76 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_0.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_1.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_42.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_63.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_69.pt 5.64 MB
waterbirds/resnet50_water

In [ ]:
# 9. Run 500-step full-epoch Rater meta-learning
os.environ["RATER_PRECOMPUTE_EMBEDDINGS_ONLY"] = "0"
shutil.rmtree(LOCAL_RUN, ignore_errors=True)
LOCAL_RUN.mkdir(parents=True, exist_ok=True)

# Rater hyperparameters live in environment variables, so the command is short.
cmd = [
    sys.executable, "-u", "main.py",
    "--dataset", "waterbirds",
    "--algorithm", "rater",
    "--backbone", "resnet50",
    "--pretrained", "True",
    "--erm_model", str(ERM_MODEL),
    "--save_folder", str(LOCAL_RUN),
    "--train_split", "train",
    "--test_split", "test",
    "--split_val", "0.4",
    "--batch_size", str(BATCH_SIZE),
    "--mode", "train",
    "--epoch", str(META_STEPS),
    "--seed", "0",
]

result = subprocess.run(cmd, cwd=str(REPO_DIR), env=os.environ.copy())
if result.returncode != 0:
    raise RuntimeError("Rater meta-training failed")

print("✓ Rater meta-training complete")

In [ ]:
# 10. Copy only model + plots + log + summary to Drive
rater_files = sorted(LOCAL_RUN.rglob("final_rater.pt"))
assert len(rater_files) == 1, f"found {len(rater_files)} final_rater.pt files"

local_exp = rater_files[0].parent
for name in ["final_rater.pt", "log.txt", "summary.txt", "plots"]:
    assert (local_exp / name).exists(), name

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_run = OUTPUT_ROOT / f"ea_gated_full_epoch_random_inner_{stamp}"
drive_run.mkdir(parents=True, exist_ok=False)

for name in ["final_rater.pt", "log.txt", "summary.txt"]:
    shutil.copy2(local_exp / name, drive_run / name)
shutil.copytree(local_exp / "plots", drive_run / "plots")

print("Saved:", drive_run)
print("Rater:", drive_run / "final_rater.pt")
print("Plots:", drive_run / "plots")

In [ ]:
# 11. Summary + plot inventory
print((drive_run / "summary.txt").read_text(encoding="utf-8"))

plots = sorted((drive_run / "plots").rglob("*.png"))
print(f"Saved {len(plots)} PNG plots")
for p in plots:
    print(p.relative_to(drive_run))